# 1. PuRe: full SST-2 BERT-base reproduction

This notebook implements the Fine-tune and PuRe rows of Table 1 for **TextFooler and TextBugger**, with all **1,821 labeled test examples**. Results are generated only by execution; no experimental result is prefilled. The default trains **18 candidates × 4 epochs** (two models × nine hyperparameters), selects on validation only, reloads both winners, evaluates clean TEST, then runs four independent attacks. This can require multiple Kaggle sessions.

Sources: [paper, §§4.2–4.5, Tables 1, 8, 12](https://aclanthology.org/2025.tacl-1.63.pdf), [official implementation](https://github.com/PuReDefence/PuRe/tree/d9ccbfbaca7a649855e690e7cc4790cbd0a80ff0), [candidate dataset and upstream provenance](https://huggingface.co/datasets/SetFit/sst2), [TextAttack recipes](https://github.com/QData/TextAttack/tree/v0.3.10/textattack/attack_recipes).

**Execution contract:** fresh Kaggle GPU runtime, Internet enabled, Tesla T4 preferred. Run cells in order; each Python cell is ordinary Python, with no shell magics. Helper definitions are separated from stage dispatch so stages can be rerun independently after setup. The first execution cell refuses local execution. Do not run training or resource downloads on a laptop. No separate training script is used.

The earlier `pure-textfooler.ipynb` informed dependency planning, atomic output writing and TensorFlow CPU isolation. Its dataset split, training duration, subset modes, query cap and per-text RNG design are deliberately excluded. Matching Table 8 sample counts does **not** establish identical raw data provenance to the authors.


## 2. Configuration and stage controls

**Recommended Kaggle workflow: separate `train`, `evaluate`, then `attack` sessions, saving and reattaching the full artifact tree between sessions.** `RUN_STAGE='all'` remains the default for compatibility, but 18 four-epoch candidates plus four full-test attacks are risky within one Kaggle session/quota; Internet and resource loading also consume session time. Select one attack job per session if needed. `paper_grid` remains enabled; staging never switches to `fixed`.

`RUN_STAGE='all'` is the default. For multiple sessions use `train`, then `evaluate`, then `attack`; `attack` requires both winning models and both complete clean evaluations. `ATTACK_JOBS` selects `all` or a list of `(model, attack)` pairs. It changes scheduling only, never the test population. Additional seeds use separate output directories by setting `SEED`; never pool their rows.

`HYPERPARAM_MODE='paper_grid'` covers learning rates `[1e-5, 2e-5, 5e-5]` and effective batch sizes `[8,16,32]`. `fixed` uses `2e-5,16` and is explicitly a protocol approximation. All candidates train four epochs with no early stopping. Ties: highest validation accuracy, then earliest epoch, then lower learning rate, then smaller effective batch size.

Unspecified paper settings chosen here: seed 42; uncased fast tokenizer, max length 128; dynamic right padding; validation/inference singleton batches; AdamW weight decay 0.01, betas (0.9,0.999), epsilon 1e-8; linear schedule, zero warmup; gradient clipping 1.0; FP32; gradient checkpointing; physical training microbatch at most 8 with accumulation to the selected effective batch. This last choice is a logged hardware accommodation, not evidence of the authors' exact batching. OOM aborts with persisted checkpoints; it never silently shrinks the selected batch or switches precision.

**Runtime recommendation:** With limited Kaggle GPU time, use `RUN_STAGE = "train"` first and persist the output before evaluation/attacks. `paper_grid` trains 18 candidates, while `fixed` trains two candidates but is explicitly an approximation. The full 1,821-sample TextFooler and TextBugger evaluations can run far longer than a single Kaggle session; do not start with `RUN_STAGE = "all"` unless prepared to save/resume.


In [1]:
from pathlib import Path
import os, sys, json, hashlib, platform, subprocess, shutil, gc, random, time, re, math
import logging, inspect, importlib, unicodedata, tempfile, zipfile, traceback, uuid
from importlib import metadata
from contextlib import contextmanager
from collections import Counter, defaultdict

RUN_STAGE = 'all'                   # all | train | evaluate | attack
HYPERPARAM_MODE = 'paper_grid'      # paper_grid | fixed (approximation)
SEED = 42                          # our implementation choice; not a claimed author seed
ATTACK_JOBS = 'all'                 # e.g. [('bert_pure', 'textfooler')]
RNG_POLICY = 'paper_like_stochastic' # deterministic_diagnostic is a separate sensitivity run
QUERY_CAP = None                    # None preserves the original recipe's unlimited budget
PREVIOUS_ARTIFACT_ROOT = None       # e.g. '/kaggle/input/prior-version/pure_reproduction'
AUTO_DETECT_PREVIOUS = True
INSTALL_DEPENDENCIES = True
DATASET_REVISION = None             # None resolves once; restored runs reuse immutable SHA
MODEL_REVISION = None
PURE_COMMIT = 'd9ccbfbaca7a649855e690e7cc4790cbd0a80ff0'
MODEL_NAME = 'bert-base-uncased'
DATASET_NAME = 'SetFit/sst2'
MODEL_IDS = ('bert_baseline', 'bert_pure')
ATTACK_NAMES = ('textfooler', 'textbugger')
ALL_JOBS = [(m, a) for m in MODEL_IDS for a in ATTACK_NAMES]
FIXED_LR, FIXED_BATCH = 2e-5, 16
MAX_LENGTH, MAX_MICROBATCH = 128, 8
EPOCHS = 4
MAX_ATTEMPTS = 3                   # infrastructure errors only; initial mismatches are never redrawn
FLUSH_EVERY = 100                  # atomic records every example; aggregate reports every 100 commits
DIAGNOSTIC_N, DIAGNOSTIC_REPEATS = 64, 5 # diagnostics only; all TEST examples are evaluated/attacked
PINS = {'transformers':'4.57.6', 'datasets':'4.4.1', 'accelerate':'1.12.0',
        'safetensors':'0.7.0', 'textattack':'0.3.10', 'tensorflow':'2.20.0',
        'tf-keras':'2.20.0', 'tensorflow-hub':'0.16.1', 'nltk':'3.9.2'}
PURE_KWARGS = dict(svd_rank=8, npc=1, center=False, num_iters=1, alpha=1.5,
                   do_pcr=True, do_pfsa=True, label_smoothing=0.0)
EXPECTED_SIZES = {'train':6920, 'validation':872, 'test':1821}
assert RUN_STAGE in {'all','train','evaluate','attack'}
assert HYPERPARAM_MODE in {'paper_grid','fixed'}
assert RNG_POLICY in {'paper_like_stochastic','deterministic_diagnostic'}
assert QUERY_CAP is None or (isinstance(QUERY_CAP,int) and QUERY_CAP >= 2)
assert SEED >= 0 and MAX_ATTEMPTS >= 1 and EPOCHS == 4
SELECTED_JOBS = ALL_JOBS if ATTACK_JOBS == 'all' else [tuple(j) for j in ATTACK_JOBS]
assert SELECTED_JOBS and len(set(SELECTED_JOBS)) == len(SELECTED_JOBS)
assert all(j in ALL_JOBS for j in SELECTED_JOBS)
if not Path('/kaggle/working').is_dir():
    raise RuntimeError('Kaggle-only execution: local work is limited to offline static/helper validation.')
ROOT = Path('/kaggle/working/pure_reproduction')
if SEED != 42 or RNG_POLICY != 'paper_like_stochastic':
    ROOT = ROOT / f'seed_{SEED}_{RNG_POLICY}'
ROOT.mkdir(parents=True, exist_ok=True)
CACHE_ROOT = Path('/kaggle/working/.pure_cache')
os.environ.update(CUDA_VISIBLE_DEVICES='0', USE_TF='0', USE_FLAX='0',
    TOKENIZERS_PARALLELISM='false', WANDB_DISABLED='true', TF_CPP_MIN_LOG_LEVEL='2',
    HF_HOME=str(CACHE_ROOT/'huggingface'), TA_CACHE_DIR=str(CACHE_ROOT/'textattack'),
    TFHUB_CACHE_DIR=str(CACHE_ROOT/'tfhub'), CUBLAS_WORKSPACE_CONFIG=':4096:8',
    TF_USE_LEGACY_KERAS='1')
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(message)s', force=True)
LOG = logging.getLogger('pure_reproduction')
LOG.addHandler(logging.FileHandler(ROOT/'execution.log'))
TRAIN_SETTINGS = dict(epochs=4, weight_decay=0.01, optimizer='adamw_torch',
    adam_beta1=0.9, adam_beta2=0.999, adam_epsilon=1e-8, scheduler='linear', warmup_ratio=0.0,
    max_grad_norm=1.0, max_microbatch=MAX_MICROBATCH, gradient_checkpointing=True,
    fp16=False, bf16=False, tf32=False, max_length=MAX_LENGTH, validation_batch_size=1,
    selection='validation accuracy descending, epoch ascending, lr ascending, batch ascending')
REQUEST = dict(schema_version=1, implementation_revision='2026-10-31-v1', seed=SEED, hyperparam_mode=HYPERPARAM_MODE,
    fixed_lr=FIXED_LR, fixed_batch=FIXED_BATCH, pure_kwargs=PURE_KWARGS, training=TRAIN_SETTINGS,
    model_name=MODEL_NAME, dataset_name=DATASET_NAME, pure_commit=PURE_COMMIT, pins=PINS,
    rng_policy=RNG_POLICY, query_cap=QUERY_CAP, inference='FP32 singleton dynamic right padding',
    max_attempts=MAX_ATTEMPTS, diagnostics_n=DIAGNOSTIC_N, diagnostics_repeats=DIAGNOSTIC_REPEATS)
LOG.info('Stage=%s seed=%s mode=%s jobs=%s', RUN_STAGE, SEED, HYPERPARAM_MODE, SELECTED_JOBS)


2026-10-31 17:58:35,630 INFO Stage=all seed=42 mode=paper_grid jobs=[('bert_baseline', 'textfooler'), ('bert_baseline', 'textbugger'), ('bert_pure', 'textfooler'), ('bert_pure', 'textbugger')]


## 3. Atomic storage, metric definitions and offline helper tests

Atomic JSON records are the authoritative state. CSVs are readable snapshots rebuilt from those records. Each record has a content checksum and fingerprints. Invalid, conflicting or foreign records cause an error, never a silent skip. A completed candidate requires hash-verified weights and four validation epochs.

Initially incorrect examples are ineligible. Eligible coverage is `(success + failed) / initially_correct`; unresolved errors, attack-time skips and pending examples prevent definitive ASR/AUA. `N_attacked` counts eligible examples with an actual attempt, including errors and skips. AvgQ uses **all these attempted examples** when their final-attempt query counts are known; successful-only AvgQ is separate. Conditional resolved-only query averages are also exported. Retry costs are reported separately, and an interrupted in-flight attempt can have unknown cost.


In [2]:
# Standard-library-only helpers: these are also exercised by local offline validation.
def stable_hash(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, ensure_ascii=False,
                                      allow_nan=False, separators=(',',':')).encode()).hexdigest()

def digest_file(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda: f.read(8*1024*1024), b''): h.update(block)
    return h.hexdigest()

def atomic_json(path, obj):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_name(path.name + '.' + uuid.uuid4().hex + '.tmp')
    with temp.open('w', encoding='utf-8') as f:
        json.dump(obj, f, indent=2, ensure_ascii=False, allow_nan=False)
        f.write('\n'); f.flush(); os.fsync(f.fileno())
    os.replace(temp, path)

def atomic_text(path,content):
    path=Path(path); path.parent.mkdir(parents=True,exist_ok=True)
    temp=path.with_name(path.name+'.'+uuid.uuid4().hex+'.tmp')
    with temp.open('w',encoding='utf-8') as f:
        f.write(content); f.flush(); os.fsync(f.fileno())
    os.replace(temp,path)

def read_json(path):
    return json.loads(Path(path).read_text(encoding='utf-8'))

def seal(payload):
    return dict(payload=payload, sha256=stable_hash(payload))

def unseal(envelope):
    assert set(envelope) == {'payload','sha256'}, 'Unexpected sealed schema'
    assert stable_hash(envelope['payload']) == envelope['sha256'], 'Artifact checksum mismatch'
    return envelope['payload']

def atomic_csv(path, frame):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_name(path.name + '.' + uuid.uuid4().hex + '.tmp')
    with temp.open('w', encoding='utf-8', newline='') as f:
        frame.to_csv(f, index=False); f.flush(); os.fsync(f.fileno())
    os.replace(temp, path)

def normalized_text(text):
    return ' '.join(unicodedata.normalize('NFKC', text).casefold().split())

def audit_dataset(split_rows, expected_sizes):
    # Content findings are nonfatal; broken sample/schema/label integrity remains fatal.
    assert set(split_rows)==set(expected_sizes), 'Missing or unexpected dataset split'
    groups = defaultdict(list)
    seen_ids = set()
    for split, rows in split_rows.items():
        assert len(rows) == expected_sizes[split], (split, len(rows))
        for r in rows:
            assert isinstance(r,dict) and {'text','label','sample_id'}<=set(r), f'Broken row schema: {split}'
            assert type(r['label']) is int and r['label'] in (0,1), f'Invalid binary label: {split}'
            assert isinstance(r['text'],str) and normalized_text(r['text']), 'Empty/non-string text'
            assert isinstance(r['sample_id'],str) and r['sample_id'], 'Missing/invalid sample ID'
            assert r['sample_id'] not in seen_ids, f'Duplicate sample ID: {r["sample_id"]}'
            seen_ids.add(r['sample_id'])
            groups[normalized_text(r['text'])].append(dict(split=split, sample_id=r['sample_id'],
                label=r['label'], original_text=r['text']))
        assert set(r['label'] for r in rows) == {0,1}, f'Missing/bad binary labels: {split}'
    overlaps = []
    for text, items in groups.items():
        if len(items) > 1:
            split_counts = dict(Counter(r['split'] for r in items))
            splits = set(split_counts)
            overlaps.append(dict(normalized_sha256=hashlib.sha256(text.encode()).hexdigest(),
                normalized_text=text, representative_original_text=items[0]['original_text'],
                occurrence_count=len(items), split_counts=split_counts, occurrences=items,
                within_split_duplicate=any(count>1 for count in split_counts.values()),
                contradictory=len({r['label'] for r in items})>1,
                across_splits=len(splits)>1, train_test_leakage={'train','test'} <= splits))
    return overlaps

def build_dataset_overlap_report(split_rows, expected_sizes):
    overlaps = audit_dataset(split_rows, expected_sizes)
    names = list(expected_sizes)
    pair_counts = {f'{left}/{right}':sum(left in g['split_counts'] and right in g['split_counts'] for g in overlaps)
        for i,left in enumerate(names) for right in names[i+1:]}
    across = sum(g['across_splits'] for g in overlaps)
    contradictory = sum(g['contradictory'] for g in overlaps)
    warnings = []
    if across:
        warnings.append(f'Potential evaluation leakage: {across} normalized-text groups occur across benchmark splits. '
            'They may affect validation selection and test estimates; preserve the original canonical splits and interpret results with caution.')
    if contradictory:
        warnings.append(f'Contradictory labels: {contradictory} normalized-text groups contain different provided labels. '
            'These are label inconsistencies or normalization collisions, not harmless duplicates; retain and disclose the original labels without guessing or relabeling.')
    return dict(normalization='NFKC + casefold + whitespace collapse; audit only', groups=overlaps,
        duplicate_groups=len(overlaps),
        within_split_duplicate_groups=sum(g['within_split_duplicate'] for g in overlaps),
        within_split_duplicate_groups_by_split={name:sum(g['split_counts'].get(name,0)>1 for g in overlaps) for name in names},
        across_split_groups=across, cross_split_group_counts=pair_counts, contradictory_groups=contradictory,
        train_test_groups=sum(g['train_test_leakage'] for g in overlaps),
        duplicate_occurrences=sum(g['occurrence_count'] for g in overlaps),
        duplicate_excess_rows=sum(g['occurrence_count']-1 for g in overlaps),
        affected_samples_by_split={name:sum(g['split_counts'].get(name,0) for g in overlaps) for name in names},
        group_counting_note='Each normalized-text group is counted once per applicable category/pair; categories can overlap. No rows are removed.',
        policy='warn_and_preserve_canonical_splits', splits_preserved=True,
        potential_evaluation_leakage=bool(across), potential_label_inconsistency=bool(contradictory), warnings=warnings)


def pct(n, d):
    return 100.0*n/d if d else None

def count_metrics(n_total, n_correct, statuses, attempted):
    assert len(statuses) == n_total and len(attempted) == n_total
    counts = Counter(statuses)
    assert set(counts) <= {'success','failed','ineligible','error','initial_mismatch','pending'}
    assert counts['ineligible'] == n_total - n_correct
    s, f = counts['success'], counts['failed']
    resolved = s+f; unresolved = n_correct-resolved
    assert 0 <= resolved <= n_correct <= n_total
    complete = unresolved == 0
    result = dict(N_total=n_total, N_initially_correct=n_correct,
        N_initially_incorrect=n_total-n_correct, N_attacked=sum(bool(a) for a in attempted),
        N_successful=s, N_failed=f, N_skipped=counts['ineligible']+counts['initial_mismatch'],
        N_skipped_eligible=counts['initial_mismatch'], N_error=counts['error'], N_pending=counts['pending'],
        N_unresolved=unresolved, eligible_coverage_pct=pct(resolved,n_correct), complete=complete,
        clean_accuracy_pct=pct(n_correct,n_total), asr_pct=pct(s,n_correct) if complete else None,
        aua_pct=pct(n_correct-s,n_total) if complete else None,
        asr_numerator=s, asr_denominator=n_correct, aua_denominator=n_total,
        aua_numerator=n_correct-s if complete else None,
        conditional_resolved_asr_pct=pct(s,resolved), conditional_resolved_denominator=resolved,
        asr_lower_pct=pct(s,n_correct), asr_upper_pct=pct(s+unresolved,n_correct),
        aua_lower_pct=pct(f,n_total), aua_upper_pct=pct(n_correct-s,n_total))
    assert s+f+counts['ineligible']+counts['error']+counts['initial_mismatch']+counts['pending'] == n_total
    if complete and n_correct:
        assert math.isclose(result['aua_pct'], result['clean_accuracy_pct']*(1-result['asr_pct']/100), abs_tol=1e-10)
    return result

def deduplicate_records(records):
    unique = {}
    for r in records:
        key = (r['run_fingerprint'],r['model'],r['attack'],r['sample_id'])
        if key in unique:
            assert stable_hash(unique[key]) == stable_hash(r), f'Conflicting duplicate: {key}'
        unique[key] = r
    return list(unique.values())

def actual_minus_paper(actual, paper):
    return None if actual is None else actual-paper

def helper_tests():
    assert normalized_text('  A\u00a0Film ') == 'a film'
    rows = {'train':[dict(text='A',label=0,sample_id='t0'),dict(text='B',label=1,sample_id='t1')],
            'validation':[dict(text='C',label=0,sample_id='v0'),dict(text='D',label=1,sample_id='v1')],
            'test':[dict(text=' a ',label=1,sample_id='e0'),dict(text='E',label=0,sample_id='e1')]}
    overlap = audit_dataset(rows, {k:2 for k in rows})
    assert overlap[0]['contradictory'] and overlap[0]['train_test_leakage']
    # A valid matching-label canonical overlap must be reported without aborting or changing rows.
    canonical={k:[dict(r) for r in v] for k,v in rows.items()}
    canonical['test'][0]['label']=0; canonical['test'][1]['label']=1
    original_hash=stable_hash(canonical)
    report=build_dataset_overlap_report(canonical,{k:2 for k in canonical})
    assert report['policy']=='warn_and_preserve_canonical_splits' and report['splits_preserved']
    assert report['duplicate_groups']==report['across_split_groups']==report['train_test_groups']==1
    assert report['cross_split_group_counts']=={'train/validation':0,'train/test':1,'validation/test':0}
    assert report['contradictory_groups']==0 and report['potential_evaluation_leakage'] and report['warnings']
    assert report['groups'][0]['representative_original_text']=='A'
    assert report['groups'][0]['occurrences']==[
        dict(split='train',sample_id='t0',label=0,original_text='A'),
        dict(split='test',sample_id='e0',label=0,original_text=' a ')]
    assert stable_hash(canonical)==original_hash, 'Audit changed canonical examples or labels'
    contradictory=build_dataset_overlap_report(rows,{k:2 for k in rows})
    assert contradictory['contradictory_groups']==1 and contradictory['potential_label_inconsistency']
    assert any('Contradictory labels:' in warning for warning in contradictory['warnings'])
    repeated={'train':[dict(text='X',label=0,sample_id='a'),dict(text='x',label=0,sample_id='b'),dict(text='Y',label=1,sample_id='c')],
              'validation':[dict(text=' X ',label=1,sample_id='d'),dict(text='Z',label=0,sample_id='e')],
              'test':[dict(text='x',label=0,sample_id='f'),dict(text='Q',label=1,sample_id='g')]}
    repeated_report=build_dataset_overlap_report(repeated,dict(train=3,validation=2,test=2))
    assert repeated_report['duplicate_groups']==1 and repeated_report['within_split_duplicate_groups']==1
    assert repeated_report['duplicate_occurrences']==4 and repeated_report['duplicate_excess_rows']==3
    assert repeated_report['affected_samples_by_split']=={'train':2,'validation':1,'test':1}
    assert all(count==1 for count in repeated_report['cross_split_group_counts'].values())
    try: audit_dataset(rows, dict(train=3,validation=2,test=2))
    except AssertionError: pass
    else: raise AssertionError('Size test failed')
    bad = {k:[dict(x) for x in v] for k,v in rows.items()}; bad['train'][0]['label']=-1
    try: audit_dataset(bad, {k:2 for k in bad})
    except AssertionError: pass
    else: raise AssertionError('Label test failed')
    missing={k:v for k,v in rows.items() if k!='train'}
    try: audit_dataset(missing,{k:2 for k in rows})
    except AssertionError: pass
    else: raise AssertionError('Missing split accepted')
    malformed={k:[dict(r) for r in v] for k,v in rows.items()}; del malformed['train'][0]['text']
    try: audit_dataset(malformed,{k:2 for k in rows})
    except AssertionError: pass
    else: raise AssertionError('Broken row schema accepted')
    m=count_metrics(5,4,['success','success','failed','failed','ineligible'],[1,1,1,1,0])
    assert m['asr_pct']==50 and m['aua_pct']==40 and m['complete']
    m=count_metrics(5,4,['success','failed','error','pending','ineligible'],[1,1,1,0,0])
    assert m['asr_pct'] is None and m['aua_pct'] is None and m['asr_lower_pct']==25 and m['asr_upper_pct']==75
    assert m['aua_lower_pct']==20 and m['aua_upper_pct']==60
    assert count_metrics(1,0,['ineligible'],[0])['asr_pct'] is None
    r=dict(run_fingerprint='x',model='b',attack='tf',sample_id='0',status='failed')
    assert len(deduplicate_records([r,r]))==1
    try: deduplicate_records([r,dict(r,status='success')])
    except AssertionError: pass
    else: raise AssertionError('Conflict was silently accepted')
    assert math.isclose(actual_minus_paper(31.0,30.37),0.63,abs_tol=1e-10)
    assert actual_minus_paper(None,30.37) is None
    assert unseal(seal(r)) == r
    with tempfile.TemporaryDirectory() as d:
        p=Path(d)/'record.json'; atomic_json(p,seal(r)); assert unseal(read_json(p))==r
    return 'Dataset, denominators, bounds, deduplication, arithmetic and atomic JSON tests passed.'

print(helper_tests())

@contextmanager
def stage(name):
    LOG.info('Starting %s', name); t=time.perf_counter()
    atomic_json(ROOT/'stage_status.json', dict(stage=name,status='running'))
    try: yield
    except BaseException as exc:
        atomic_json(ROOT/'stage_status.json', dict(stage=name,status='interrupted_or_failed',
            error=repr(exc),elapsed_seconds=time.perf_counter()-t))
        LOG.exception('Stage failed: %s',name); raise
    else:
        atomic_json(ROOT/'stage_status.json',dict(stage=name,status='complete',elapsed_seconds=time.perf_counter()-t))


Dataset, denominators, bounds, deduplication, arithmetic and atomic JSON tests passed.


## 4. True cross-session resume and dependency preflight

`/kaggle/working` survives an interruption **only while the session/output still exists**. Save a Kaggle notebook version with outputs, download/archive them if needed, then create or attach a Kaggle input dataset containing the **entire `pure_reproduction/` tree** (including models, candidates, clean ledgers and attack ledgers). Attach that dataset to a new notebook session. The compact report ZIP excludes the state needed to resume training/attacks.

Mounted prior roots are detected from `experiment_config.json`. An explicit `PREVIOUS_ARTIFACT_ROOT` resolves ambiguity. Matching configuration, immutable Hub revisions, data hashes, source hashes, environment identity, checkpoint hashes and per-record fingerprints are required before state is copied. Distinct seeds/policies cannot merge. Conflicting copies fail; choose the desired saved lineage explicitly. For one-job sessions, restore the latest cumulative tree each time. Keep a saved version before the session quota expires; a hard runtime kill cannot execute Python cleanup, but all committed per-example records remain recoverable from saved outputs.

The protocol implementation revision is stored in `REQUEST`; bump it for scientific changes to notebook helpers. Pinned dependencies are a reproducibility environment choice, not the paper's unspecified library versions. Preserve Kaggle's torch/CUDA, NumPy, SciPy, pandas and scikit-learn. Pip resolves a plan before installation and refuses to replace imported packages. TensorFlow/USE stays on CPU so PyTorch has the T4. Required imports, CUDA, NLTK, embeddings, POS, USE and both complete recipe searches must work before expensive training; there is no weaker fallback.


In [3]:
# Discover state without modifying mounted inputs or mixing different experiments.
prior_candidates = []
if PREVIOUS_ARTIFACT_ROOT:
    prior_candidates = [Path(PREVIOUS_ARTIFACT_ROOT)]
elif AUTO_DETECT_PREVIOUS:
    prior_candidates = sorted({p.parent for p in Path('/kaggle/input').rglob('experiment_config.json')})
matching_roots = []
for p in [ROOT, *prior_candidates]:
    cfg_path=p/'experiment_config.json'
    if cfg_path.is_file():
        c=read_json(cfg_path)
        if c.get('request') == REQUEST:
            if DATASET_REVISION and c['dataset_revision'] != DATASET_REVISION: continue
            if MODEL_REVISION and c['model_revision'] != MODEL_REVISION: continue
            matching_roots.append(p)
        elif p == ROOT or PREVIOUS_ARTIFACT_ROOT:
            raise RuntimeError(f'Configuration differs from {p}; use a separate output lineage/runtime.')
if ROOT in matching_roots:
    PRIOR_ROOT=ROOT
else:
    if len(matching_roots)>1:
        raise RuntimeError(f'Multiple prior lineages: {matching_roots}. Set PREVIOUS_ARTIFACT_ROOT explicitly.')
    PRIOR_ROOT=matching_roots[0] if matching_roots else None
if PREVIOUS_ARTIFACT_ROOT and PRIOR_ROOT is None:
    raise RuntimeError('Explicit prior root does not contain matching experiment_config.json')
PRIOR_CONFIG=read_json(PRIOR_ROOT/'experiment_config.json') if PRIOR_ROOT else None
if PRIOR_CONFIG:
    DATASET_REVISION=PRIOR_CONFIG['dataset_revision']; MODEL_REVISION=PRIOR_CONFIG['model_revision']
LOG.info('Prior artifacts: %s; mounted roots discovered: %d',PRIOR_ROOT,len(prior_candidates))

def installed_version(name):
    try: return metadata.version(name)
    except metadata.PackageNotFoundError: return None

with stage('dependencies: resolve before install'):
    protected=('torch','torchvision','torchaudio','numpy','scipy','pandas','scikit-learn')
    protected_versions={p:installed_version(p) for p in protected if installed_version(p)}
    assert 'torch' in protected_versions, 'Use a Kaggle image with CUDA-enabled PyTorch'
    if INSTALL_DEPENDENCIES:
        constraints=ROOT/'environment_constraints.txt'
        constraints.write_text(''.join(f'{p}=={v}\n' for p,v in protected_versions.items()))
        args=[sys.executable,'-m','pip','install','--disable-pip-version-check','--constraint',str(constraints),
              *[f'{p}=={v}' for p,v in PINS.items()]]
        plan_path=ROOT/'pip_plan.json'
        plan=subprocess.run([*args,'--dry-run','--report',str(plan_path)],capture_output=True,text=True)
        (ROOT/'dependency_plan.log').write_text(plan.stdout+'\n'+plan.stderr)
        if plan.returncode:
            raise RuntimeError('Dependency resolution failed; inspect dependency_plan.log in Outputs. Use a compatible fresh Kaggle image.')
        distributions=metadata.packages_distributions()
        norm=lambda x: re.sub(r'[-_.]+','-',x).lower()
        for item in read_json(plan_path).get('install',[]):
            name,target=item['metadata']['name'],item['metadata']['version']
            roots=[k for k,v in distributions.items() if any(norm(d)==norm(name) for d in v)]
            if installed_version(name)!=target and any(k in sys.modules for k in roots):
                raise RuntimeError(f'{name} already imported and would change: restart in a fresh Kaggle session')
        subprocess.run(args,check=True)
    for p,v in PINS.items(): assert installed_version(p)==v, (p,installed_version(p),v)
    for p,v in protected_versions.items(): assert installed_version(p)==v, f'Protected package changed: {p}'


2026-10-31 17:58:35,739 INFO Prior artifacts: None; mounted roots discovered: 0
2026-10-31 17:58:35,740 INFO Starting dependencies: resolve before install


  Using cached transformers-4.57.6-py3-none-any.whl.metadata (43 kB)
  Using cached datasets-4.4.1-py3-none-any.whl.metadata (19 kB)
  Using cached accelerate-1.12.0-py3-none-any.whl.metadata (19 kB)
  Using cached safetensors-0.7.0-cp38-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (4.1 kB)
  Using cached textattack-0.3.10-py3-none-any.whl.metadata (38 kB)
  Using cached tf_keras-2.20.0-py3-none-any.whl.metadata (1.8 kB)
  Using cached nltk-3.9.2-py3-none-any.whl.metadata (3.2 kB)
  Using cached huggingface_hub-0.36.2-py3-none-any.whl.metadata (15 kB)
  Using cached tokenizers-0.22.2-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (7.3 kB)
  Using cached dill-0.4.0-py3-none-any.whl.metadata (10 kB)
  Using cached multiprocess-0.70.18-py313-none-any.whl.metadata (7.2 kB)
  Using cached fsspec-2025.10.0-py3-none-any.whl.metadata (10 kB)
  Using cached bert_score-0.3.13-py3-none-any.whl.metadata (15 kB)
  Using cached flair-0.15.1-py3-none-any.whl.metadat

Reason for being yanked: <none given>


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 22.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 27.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 30.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 13.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 45.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 30.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 769.7/769.7 kB 21.2 MB/s  0:00:00
  Created wheel for pinyin: filename=pinyin-0.4.0-py3-none-any.whl size=3630548 sha256=0e2b50762b586f0a7d3e12d2d286cc4651d5e5f2bb5b5bf675d6006daaf97fd2
  Stored in directory: /root/.cache/pip/wheels/45/23/be/670fc418eadc93220baf097682c12d3bd6210e28e080317c9a
  Created wheel for langdetect: filename=langdetect-1.0.9-py3-none-any.whl size=993332 sha256=da3c3df82db12888d94e771e122348e82e132b8e0c4ec788cef3e19c4b6d36a5
  Stored in directory: /root/.cache/pip/wheels/eb/87/2

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.48.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
pathos 0.3.5 requires dill>=0.4.1, but you have dill 0.4.0 which is incompatible.
pathos 0.3.5 requires multiprocess>=0.70.19, but you have multiprocess 0.70.18 which is incompatible.
s3fs 2025.12.0 requires fsspec==2025.12.0, but you have fsspec 2025.10.0 which is incompatible.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.26.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.40.0 requires safetensors>=0.8.

In [4]:
with stage('dependencies: imports and hardware'):
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    import torch
    import transformers
    import tensorflow as tf
    tf.config.set_visible_devices([], 'GPU')
    tf.config.threading.set_intra_op_parallelism_threads(2)
    tf.config.threading.set_inter_op_parallelism_threads(2)
    import tensorflow_hub as hub
    import nltk
    from huggingface_hub import HfApi, snapshot_download
    from datasets import load_dataset
    from safetensors import safe_open
    from transformers import (AutoConfig, AutoTokenizer, BertForSequenceClassification,
        Trainer, TrainingArguments, TrainerCallback, DataCollatorWithPadding, set_seed)
    from transformers.trainer_utils import get_last_checkpoint
    from sklearn.metrics import accuracy_score, precision_recall_fscore_support, f1_score, confusion_matrix
    from tqdm.auto import tqdm
    from IPython.display import display, FileLink
    for resource in ('averaged_perceptron_tagger_eng','averaged_perceptron_tagger',
                     'universal_tagset','punkt_tab','punkt','stopwords','wordnet','omw-1.4'):
        if not nltk.download(resource,quiet=True,raise_on_error=True):
            raise RuntimeError(f'NLTK resource unavailable: {resource}')
    import textattack
    from textattack.models.wrappers import ModelWrapper
    from textattack.attack_recipes import TextFoolerJin2019, TextBuggerLi2018
    from textattack.attack_results import SuccessfulAttackResult, FailedAttackResult, SkippedAttackResult
    from textattack.goal_functions import UntargetedClassification
    from textattack.shared import AttackedText
    import flair
    flair.device=torch.device('cpu')
    assert tuple(int(x) for x in torch.__version__.split('+')[0].split('.')[:2]) >= (2,6), 'Trainer safe optimizer/RNG resume requires torch>=2.6; use a fresh supported Kaggle image'
    assert torch.cuda.is_available(), 'Enable Kaggle GPU before any training'
    DEVICE=torch.device('cuda:0')
    assert torch.tensor([1.,2.],device=DEVICE).sum().item()==3.
    assert tf.reduce_sum(tf.constant([1.,2.])).numpy()==3.
    assert not tf.config.get_visible_devices('GPU')
    torch.backends.cuda.matmul.allow_tf32=False
    torch.backends.cudnn.allow_tf32=False
    torch.backends.cudnn.benchmark=False
    torch.backends.cudnn.deterministic=True
    torch.use_deterministic_algorithms(RNG_POLICY=='deterministic_diagnostic')
    # PYTHONHASHSEED set inside a running kernel is ineffective. Record rather than claim otherwise.
    set_seed(SEED)
    SOFTWARE={p:installed_version(p) for p in sorted(set(PINS)|set(protected)|
        {'tokenizers','huggingface-hub','flair','sentence-transformers','protobuf','keras','matplotlib'})}
    ENVIRONMENT=dict(python=platform.python_version(),platform=platform.platform(),packages=SOFTWARE,
        cuda=torch.version.cuda,gpu=torch.cuda.get_device_name(0),gpu_count=torch.cuda.device_count(),
        gpu_memory_bytes=torch.cuda.get_device_properties(0).total_memory,
        pythonhashseed=os.environ.get('PYTHONHASHSEED'),strict_deterministic_kernels=torch.are_deterministic_algorithms_enabled(),
        notebook_static_validation_only_locally=True)
    ENV_IDENTITY=dict(python=platform.python_version(),packages=SOFTWARE,cuda=torch.version.cuda,
                      gpu=torch.cuda.get_device_name(0))
    (ROOT/'pip_freeze.txt').write_text(subprocess.check_output([sys.executable,'-m','pip','freeze'],text=True))
    checked=subprocess.run([sys.executable,'-m','pip','check'],capture_output=True,text=True)
    (ROOT/'pip_check.txt').write_text(checked.stdout+checked.stderr)
    # Image-wide conflicts are logged; functional required-resource checks below are mandatory.
    atomic_json(ROOT/'environment_info.json',ENVIRONMENT)
    LOG.info('Hardware: %s; memory %.2f GiB',ENVIRONMENT['gpu'],ENVIRONMENT['gpu_memory_bytes']/2**30)


2026-10-31 17:59:51,394 INFO Starting dependencies: imports and hardware
2026-10-31 17:59:51,778 INFO NumExpr defaulting to 4 threads.
/usr/local/lib/python3.13/dist-packages/tensorflow_hub/__init__.py:61: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import parse_version
2026-10-31 18:00:20,718 INFO Disabling Tensorflow because USE_TORCH is set
2026-10-31 18:00:20,720 INFO JAX version 0.11.1 available.
textattack: Updating TextAttack package dependencies.
textattack: Downloading NLTK required packages.
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /usr/share/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package stopwords to /usr/share/nltk_data...
[nltk_data] 

## 5. Immutable official source and untouched pretrained BERT

The preferred official commit is required; if GitHub cannot serve it, execution fails with a clear error. Selecting another commit requires an explicit edited configuration and documented justification. Neither PCR, PFSA, mean pooling nor classifier code is patched. Constructor arguments and resolved module attributes are checked against source signatures.

Official defaults are rank 5 and alpha 1.0; this experiment uses rank 8 and alpha 1.5 corresponding to the supplied paper target. Baseline uses BERT's standard pooling/classifier; official PuRe disables BERT's pooler and has its own pooling/classifier. Shared **embeddings and encoder** must be identical to untouched pretrained weights; the absent PuRe pooler and newly initialized task heads are expected. Unexpected missing backbone tensors, shape mismatches or unexpected backbone keys are rejected.

Backbone validation compares checkpoint **learned embeddings/encoder parameters** numerically and compares the two independently loaded models. Non-trainable positional/token-type buffers are diagnosed separately, rather than causing a misleading raw-`state_dict` hash failure.


In [5]:
with stage('pinned source and pretrained snapshot'):
    SOURCE_DIR=ROOT/'official_source'
    SOURCE_DIR.mkdir(exist_ok=True)
    # Fetch exact git objects in a disposable cache, then copy only the two unmodified source files.
    repo=CACHE_ROOT/'PuRe_git'
    if not (repo/'.git').is_dir():
        repo.mkdir(parents=True,exist_ok=True)
        subprocess.run(['git','init',str(repo)],check=True,capture_output=True)
        subprocess.run(['git','-C',str(repo),'remote','add','origin','https://github.com/PuReDefence/PuRe.git'],check=True)
    origin=subprocess.check_output(['git','-C',str(repo),'remote','get-url','origin'],text=True).strip()
    assert origin=='https://github.com/PuReDefence/PuRe.git'
    subprocess.run(['git','-C',str(repo),'fetch','--depth','1','origin',PURE_COMMIT],check=True)
    fetched=subprocess.check_output(['git','-C',str(repo),'rev-parse','FETCH_HEAD'],text=True).strip()
    assert fetched==PURE_COMMIT
    for name in ('pure.py','modeling_bert.py'):
        content=subprocess.check_output(['git','-C',str(repo),'show',f'{PURE_COMMIT}:{name}'])
        p=SOURCE_DIR/name
        if p.exists(): assert p.read_bytes()==content, f'Modified official source: {p}'
        p.write_bytes(content)
    SOURCE_HASHES={n:digest_file(SOURCE_DIR/n) for n in ('pure.py','modeling_bert.py')}
    for name in ('pure','modeling_bert'):
        if name in sys.modules:
            assert Path(sys.modules[name].__file__).resolve()==(SOURCE_DIR/f'{name}.py').resolve(), 'Stale official module; restart kernel'
    sys.path.insert(0,str(SOURCE_DIR))
    from modeling_bert import PureBertForSequenceClassification
    signature=inspect.signature(PureBertForSequenceClassification.__init__)
    assert set(PURE_KWARGS) <= set(signature.parameters), str(signature)
    IMPLEMENTATION_DEFAULTS={k:signature.parameters[k].default for k in PURE_KWARGS}
    assert IMPLEMENTATION_DEFAULTS['svd_rank']==5 and IMPLEMENTATION_DEFAULTS['alpha']==1.0
    MODEL_COMMIT=HfApi().model_info(MODEL_NAME,revision=MODEL_REVISION or 'main').sha
    assert re.fullmatch('[0-9a-f]{40}',MODEL_COMMIT)
    PRETRAINED_DIR=Path(snapshot_download(MODEL_NAME,revision=MODEL_COMMIT,allow_patterns=[
        'config.json','model.safetensors','tokenizer.json','tokenizer_config.json','special_tokens_map.json','vocab.txt']))
    assert (PRETRAINED_DIR/'model.safetensors').is_file()
    PRETRAINED_HASH=digest_file(PRETRAINED_DIR/'model.safetensors')
    tokenizer=AutoTokenizer.from_pretrained(PRETRAINED_DIR,local_files_only=True)
    assert tokenizer.is_fast and tokenizer.do_lower_case and tokenizer.padding_side=='right'
    assert tokenizer('preflight tokenization',max_length=MAX_LENGTH,truncation=True)['input_ids']
    atomic_json(ROOT/'source_provenance.json',dict(repository='https://github.com/PuReDefence/PuRe',
        commit=PURE_COMMIT,source_sha256=SOURCE_HASHES,constructor_signature=str(signature),
        implementation_defaults=IMPLEMENTATION_DEFAULTS,requested_values=PURE_KWARGS,
        pretrained_model=MODEL_NAME,pretrained_commit=MODEL_COMMIT,weights_sha256=PRETRAINED_HASH,
        pretrained_config_sha256=digest_file(PRETRAINED_DIR/'config.json')))

def free_gpu():
    gc.collect(); torch.cuda.empty_cache()

def tensor_hash(t):
    a=t.detach().cpu().contiguous().numpy()
    return hashlib.sha256(a.tobytes()).hexdigest()

with safe_open(str(PRETRAINED_DIR/'model.safetensors'),framework='pt',device='cpu') as handle:
    pretrained_keys=set(handle.keys())
    SHARED_PRETRAINED={k.removeprefix('bert.'):tensor_hash(handle.get_tensor(k))
        for k in sorted(pretrained_keys) if k.startswith(('bert.embeddings.','bert.encoder.'))}
assert SHARED_PRETRAINED, 'Pretrained file lacks BERT backbone'
INITIAL_SHARED_HASH=stable_hash(SHARED_PRETRAINED)

def verify_pure_settings(model):
    p=model.pure
    resolved=dict(svd_rank=p.target_rank,npc=p.npc,center=p.center,num_iters=p.num_iters,
        alpha=p.attention.alpha,do_pcr=p.do_pcr,do_pfsa=p.do_pfsa,label_smoothing=model.label_smoothing)
    assert resolved==PURE_KWARGS, resolved
    assert model.bert.pooler is None
    return resolved

FIRST_LOADED_BACKBONE_HASH = None

def new_model(model_id):
    assert model_id in MODEL_IDS
    set_seed(SEED)  # independent fresh initializations; never read a fine-tuned model here
    config=AutoConfig.from_pretrained(PRETRAINED_DIR,local_files_only=True,num_labels=2,
        id2label={0:'negative',1:'positive'},label2id={'negative':0,'positive':1})
    cls=BertForSequenceClassification if model_id=='bert_baseline' else PureBertForSequenceClassification
    kwargs={} if model_id=='bert_baseline' else PURE_KWARGS
    model,loading=cls.from_pretrained(PRETRAINED_DIR,config=config,local_files_only=True,
        output_loading_info=True,**kwargs)
    allowed_missing={'classifier.weight','classifier.bias'}
    assert set(loading.get('missing_keys',[])) <= allowed_missing, loading
    assert not loading.get('mismatched_keys') and not loading.get('error_msgs'), loading
    allowed_unexpected=lambda k: k.startswith('cls.') or (model_id=='bert_pure' and k.startswith('bert.pooler.'))
    assert all(allowed_unexpected(k) for k in loading.get('unexpected_keys',[])), loading
    # Compare actual *learned parameters*, not every state_dict entry.  Some
    # BERT implementations expose positional/token-type buffers in state_dict
    # that are absent from the original safetensors snapshot (or vice versa).
    # Those buffers are not pretrained learned backbone weights.
    shared_parameters = {k: p.detach().cpu() for k, p in model.bert.named_parameters()
                         if k.startswith(('embeddings.', 'encoder.'))}
    shared = {k: tensor_hash(v) for k, v in shared_parameters.items()}
    assert shared, 'Loaded BERT has no shared backbone parameters'
    with safe_open(str(PRETRAINED_DIR/'model.safetensors'), framework='pt', device='cpu') as checkpoint:
        def original_key(k):
            # Backward compatibility with old HF LayerNorm gamma/beta naming.
            k = k.removeprefix('bert.')
            if '.LayerNorm.gamma' in k: k = k.replace('.LayerNorm.gamma', '.LayerNorm.weight')
            if '.LayerNorm.beta' in k: k = k.replace('.LayerNorm.beta', '.LayerNorm.bias')
            return k
        checkpoint_keys = {original_key(k): k for k in checkpoint.keys()
                           if k.startswith(('bert.embeddings.', 'bert.encoder.'))}
        missing = sorted(set(shared) - set(checkpoint_keys))
        extras = sorted(set(checkpoint_keys) - set(shared))
        # These are deterministic positional buffers, not learned parameters.
        benign_buffers = {'embeddings.position_ids', 'embeddings.token_type_ids'}
        unexpected_extra = sorted(set(extras) - benign_buffers)
        discrepancies = []
        dtype_conversions = []
        for name, loaded in shared_parameters.items():
            if name not in checkpoint_keys: continue
            original = checkpoint.get_tensor(checkpoint_keys[name])
            if tuple(loaded.shape) != tuple(original.shape):
                discrepancies.append(dict(name=name, reason='shape', original_shape=list(original.shape),
                                          loaded_shape=list(loaded.shape)))
                continue
            if loaded.dtype == original.dtype:
                equal = torch.equal(loaded, original)
            else:
                # Check the documented dtype conversion, not raw bytes.  On a
                # canonical FP32 BERT this branch should never be needed.
                converted = original.to(loaded.dtype)
                equal = torch.equal(loaded, converted)
                dtype_conversions.append(dict(name=name, checkpoint_dtype=str(original.dtype),
                                              loaded_dtype=str(loaded.dtype)))
            if not equal:
                delta = (loaded.float() - original.float()).abs()
                discrepancies.append(dict(name=name, reason='value',
                    original_dtype=str(original.dtype), loaded_dtype=str(loaded.dtype),
                    max_abs_diff=float(delta.max().item()),
                    differing_elements=int((loaded.float() != original.float()).sum().item())))
        summary = dict(model=model_id, checked_trainable_parameter_tensors=len(shared),
                       missing_learned_parameter_keys=missing,
                       unused_checkpoint_backbone_keys=extras,
                       unexpected_checkpoint_backbone_keys=unexpected_extra,
                       dtype_conversions=dtype_conversions,
                       mismatches=discrepancies[:30], total_mismatches=len(discrepancies),
                       baseline_or_pure_loaded_backbone_sha256=stable_hash(shared),
                       original_checkpoint_backbone_sha256=INITIAL_SHARED_HASH)
        atomic_json(ROOT/f'backbone_diagnostics_{model_id}.json', summary)
        LOG.info('%s: verified %d source-learned tensors; source-only buffer keys=%s; dtype conversions=%d; mismatches=%d',
                 model_id,len(shared),extras,len(dtype_conversions),len(discrepancies))
        if missing or unexpected_extra or discrepancies:
            raise RuntimeError('Genuine pretrained backbone mismatch: inspect backbone_diagnostics_'
                               + model_id + '.json (missing keys, shape/dtype and element-level differences)')
    global FIRST_LOADED_BACKBONE_HASH
    loaded_hash = stable_hash(shared)
    if FIRST_LOADED_BACKBONE_HASH is None:
        FIRST_LOADED_BACKBONE_HASH = loaded_hash
    elif FIRST_LOADED_BACKBONE_HASH != loaded_hash:
        raise RuntimeError('BERT and PuRe began with different pretrained backbone weights')
    if model_id=='bert_pure': verify_pure_settings(model)
    atomic_json(ROOT/f'initialization_{model_id}.json',dict(loading_info=loading,
        shared_backbone_sha256=INITIAL_SHARED_HASH,expected_new_heads=sorted(allowed_missing),
        dropout=dict(hidden=config.hidden_dropout_prob,attention=config.attention_probs_dropout_prob,
                     classifier=model.dropout.p)))
    LOG.info('%s: expected new task head; verified shared learned backbone hash %s',model_id,loaded_hash)
    return model


2026-10-31 18:00:34,691 INFO Starting pinned source and pretrained snapshot
From https://github.com/PuReDefence/PuRe
 * branch            d9ccbfbaca7a649855e690e7cc4790cbd0a80ff0 -> FETCH_HEAD


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

vocab.txt: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

## 6. Exact full-sentence SST-2 data and leakage audit

Load `SetFit/sst2` at its resolved immutable Hub revision; assert exactly **6,920 / 872 / 1,821** train/validation/test rows and binary labels 0=negative, 1=positive. Original texts, original row order and split-qualified sample IDs are preserved. Normalization (NFKC, casefold, whitespace collapse) is used only for auditing, never to change model inputs.

All repeated normalized texts, including duplicates within splits, are reported with exact group counts, sample IDs, split names, labels and original-text representatives. Cross-split duplicates warn about potential evaluation leakage; contradictory labels receive a separate warning about label inconsistencies or normalization collisions. These content findings do not automatically abort a valid canonical dataset. The original 6,920/872/1,821 rows, labels, split membership and order are retained without removal, deduplication or relabeling. Sizes, schema, labels, sample IDs and immutable-revision checks remain fatal for genuine integrity failures. The dataset card points to Academia Sinica's sentiment dataset; matching counts alone cannot prove equivalence to the authors' raw files.


In [6]:
with stage('dataset: immutable revision and complete split audit'):
    DATASET_COMMIT=HfApi().dataset_info(DATASET_NAME,revision=DATASET_REVISION or 'main').sha
    assert re.fullmatch('[0-9a-f]{40}',DATASET_COMMIT)
    raw_data=load_dataset(DATASET_NAME,revision=DATASET_COMMIT)
    assert set(raw_data)==set(EXPECTED_SIZES), list(raw_data)
    SPLIT_ROWS={}
    split_info={}
    for split,n in EXPECTED_SIZES.items():
        ds=raw_data[split]
        assert len(ds)==n and {'text','label'}<=set(ds.column_names), (split,len(ds),ds.column_names)
        rows=[]
        for i,r in enumerate(ds):
            assert isinstance(r['label'],(int,np.integer)) and not isinstance(r['label'],bool)
            label=int(r['label']); assert label in (0,1), (split,i,label)
            if 'label_text' in r:
                assert str(r['label_text']).casefold()=={0:'negative',1:'positive'}[label], r
            text=r['text']; assert isinstance(text,str) and normalized_text(text)
            rows.append(dict(sample_id=f'sst2:{split}:{i:05d}',source_index=i,text=text,label=label,
                original_text_sha256=hashlib.sha256(text.encode()).hexdigest()))
        feature_names=getattr(ds.features['label'],'names',None)
        if feature_names is not None: assert feature_names==['negative','positive'], feature_names
        SPLIT_ROWS[split]=rows
        split_info[split]=dict(size=n,hf_fingerprint=ds._fingerprint,
            content_sha256=stable_hash(rows),label_counts=dict(Counter(str(r['label']) for r in rows)))
    audit=build_dataset_overlap_report(SPLIT_ROWS,EXPECTED_SIZES)
    overlap=audit['groups']
    atomic_json(ROOT/'dataset_overlap_report.json',audit)
    print(f'SST-2 overlap audit: {audit["duplicate_groups"]} normalized duplicate groups; '
          f'{audit["within_split_duplicate_groups"]} with within-split repeats; '
          f'{audit["across_split_groups"]} across splits; {audit["contradictory_groups"]} contradictory-label groups.')
    print('Cross-split group counts: '+', '.join(f'{pair}={count}' for pair,count in audit['cross_split_group_counts'].items()))
    print('Original benchmark splits retained: train=6920, validation=872, test=1821. '
          'Full groups, sample IDs, labels and original texts: dataset_overlap_report.json')
    for warning in audit['warnings']:
        LOG.warning(warning)
    DATA_MANIFEST=dict(name=DATASET_NAME,revision=DATASET_COMMIT,
        source_url=f'https://huggingface.co/datasets/{DATASET_NAME}/tree/{DATASET_COMMIT}',
        card_url=f'https://huggingface.co/datasets/{DATASET_NAME}/blob/{DATASET_COMMIT}/README.md',
        upstream_card_provenance='https://github.com/AcademiaSinicaNLPLab/sentiment_dataset/tree/master/data',
        label_mapping={'0':'negative','1':'positive'},splits=split_info,
        rows=SPLIT_ROWS,overlap_summary={k:v for k,v in audit.items() if k!='groups'},
        potential_evaluation_leakage=audit['potential_evaluation_leakage'],
        potential_label_inconsistency=audit['potential_label_inconsistency'],
        dataset_warnings=audit['warnings'],split_preservation_policy=audit['policy'],
        provenance_caveat='Matching Table 8 counts does not prove identical raw data provenance to the authors.')
    atomic_json(ROOT/'dataset_manifest.json',DATA_MANIFEST)
    # Content findings are disclosed above; canonical benchmark rows are never modified or repartitioned.
    truncation={}
    for split,rows in SPLIT_ROWS.items():
        lengths=[len(tokenizer(r['text'],truncation=False)['input_ids']) for r in rows]
        truncation[split]=dict(max_token_length=max(lengths),examples_over_max_length=sum(x>MAX_LENGTH for x in lengths),
                               fraction_over_max_length=sum(x>MAX_LENGTH for x in lengths)/len(lengths))
    atomic_json(ROOT/'tokenization_audit.json',dict(max_length=MAX_LENGTH,splits=truncation,
        note='Descriptive audit only; no test-dependent change to max_length is allowed'))
    TEST_ROWS=SPLIT_ROWS['test']; SAMPLE_BY_ID={r['sample_id']:r for r in TEST_ROWS}
    assert len(TEST_ROWS)==1821 and len(SAMPLE_BY_ID)==1821
    LOG.info('Dataset pinned %s: %s; audit groups=%s',DATASET_COMMIT,EXPECTED_SIZES,len(overlap))


2026-10-31 18:00:41,391 INFO Starting dataset: immutable revision and complete split audit


README.md:   0%|          | 0.00/378 [00:00<?, ?B/s]

Repo card metadata block was not found. Setting CardData to empty.
2026-10-31 18:00:41,555 WARNING Repo card metadata block was not found. Setting CardData to empty.


train.jsonl:   0%|          | 0.00/1.07M [00:00<?, ?B/s]

dev.jsonl:   0%|          | 0.00/136k [00:00<?, ?B/s]

test.jsonl:   0%|          | 0.00/281k [00:00<?, ?B/s]

... (2 outputs left out)

Generating test split: 0 examples [00:00, ? examples/s]

2026-10-31 18:00:43,032 WARNING Potential evaluation leakage: 2 normalized-text groups occur across benchmark splits. They may affect validation selection and test estimates; preserve the original canonical splits and interpret results with caution.


SST-2 overlap audit: 10 normalized duplicate groups; 9 with within-split repeats; 2 across splits; 0 contradictory-label groups.
Cross-split group counts: train/validation=0, train/test=2, validation/test=0
Original benchmark splits retained: train=6920, validation=872, test=1821. Full groups, sample IDs, labels and original texts: dataset_overlap_report.json


2026-10-31 18:00:44,277 INFO Dataset pinned 00ea8ccb7a54b4e3780a3e51aa3f80361ff849c0: {'train': 6920, 'validation': 872, 'test': 1821}; audit groups=10


## 7. Official TextAttack recipes and mandatory end-to-end preflight

Build `TextFoolerJin2019` and `TextBuggerLi2018` directly. Record transformation, every constraint (canonicalizing Python object addresses in metadata only), search method, model cache settings, source checksums and original query budget before applying an optional explicitly labeled cap. The default query budget is unlimited, not 500. Word-importance search and semantic constraints are preserved. TextBugger is the recipe's black-box variant.

Pinned API review: [Trainer callbacks/save order and accumulation](https://github.com/huggingface/transformers/blob/v4.57.6/src/transformers/trainer.py), [TextAttack goal-function query/cache behavior](https://github.com/QData/TextAttack/blob/v0.3.10/textattack/goal_functions/goal_function.py), and [native result query counters](https://github.com/QData/TextAttack/blob/v0.3.10/textattack/attack_results/attack_result.py). Runtime probes check recipe transformations/thresholds, normalized wrapper probabilities, cached logical-versus-physical queries and save/reload before the grid. These checks remain pending until Kaggle execution.

The wrapper returns probabilities, processes texts in their received order with singleton forwards, and seeds only at stage/job boundaries. TextAttack's default model-result caching remains enabled **within** an example; caches are cleared between examples (also necessary for reproducible resume). Logical TextAttack query counts include the clean query and can differ from physical wrapper calls due to caching. Both are recorded. Singleton forwarding is an explicit batching choice because official PuRe mean pooling includes zero-padded positions; it does not replace or patch that pooling. Runtime and accuracy can differ from a padded-batch evaluation.


In [7]:
def capture_rng():
    n=np.random.get_state()
    return dict(python=random.getstate(),numpy=[n[0],n[1].tolist(),int(n[2]),int(n[3]),float(n[4])],
        torch_cpu=torch.get_rng_state().tolist(),cuda=[s.tolist() for s in torch.cuda.get_rng_state_all()])

def tuple_tree(v):
    return tuple(tuple_tree(x) for x in v) if isinstance(v,list) else v

def restore_rng(state):
    random.setstate(tuple_tree(state['python']))
    n=state['numpy']; np.random.set_state((n[0],np.array(n[1],dtype=np.uint32),n[2],n[3],n[4]))
    torch.set_rng_state(torch.tensor(state['torch_cpu'],dtype=torch.uint8))
    assert len(state['cuda'])==torch.cuda.device_count(), 'GPU topology changed; cannot restore exact RNG state'
    torch.cuda.set_rng_state_all([torch.tensor(x,dtype=torch.uint8) for x in state['cuda']])

@contextmanager
def preserve_rng():
    before=capture_rng()
    try: yield
    finally: restore_rng(before)

class VictimWrapper(ModelWrapper):
    def __init__(self,model,tokenizer,policy=RNG_POLICY):
        self.model,self.tokenizer,self.policy=model,tokenizer,policy
        self.model.eval(); self.physical_queries=0; self.last_logits=None
    def __call__(self,texts):
        values=[]
        for text in texts:
            assert isinstance(text,str), 'SST-2 requires one plain-text input'
            # Both policies consume one continuous run-level stream. Neither seeds a text.
            values.append(self._one(text))
        return np.vstack(values) if values else np.empty((0,2),dtype=np.float32)
    def _one(self,text):
        def forward():
            tokens=self.tokenizer(text,max_length=MAX_LENGTH,truncation=True,padding=True,return_tensors='pt')
            tokens={k:v.to(DEVICE) for k,v in tokens.items()}
            self.physical_queries+=1
            with torch.inference_mode():
                logits=self.model(**tokens,return_dict=True).logits
            assert tuple(logits.shape)==(1,2) and torch.isfinite(logits).all().item(), 'Invalid victim logits'
            self.last_logits=logits.detach().float().cpu().numpy()[0]
            return logits.softmax(-1).float().cpu().numpy()
        return forward()

RECIPES={'textfooler':TextFoolerJin2019,'textbugger':TextBuggerLi2018}
RECIPE_MANIFEST={}
def canonical_repr(component):
    # Object addresses are runtime metadata, not scientific parameters. Strip only
    # Python's address suffix so identical defaults fingerprint identically after restart.
    return re.sub(r' at 0x[0-9a-fA-F]+(?=>)', '', repr(component))

def describe_component(component):
    description={'class':type(component).__name__,'repr':canonical_repr(component)}
    if hasattr(component,'_tfhub_url'): description['tfhub_url']=component._tfhub_url
    # Canonicalize explicit set/list parameters; never change the component itself.
    for key in ('stopwords','matching_column_labels','columns_to_ignore'):
        value=getattr(component,key,None)
        if isinstance(value,(set,list,tuple)):
            description[key]=sorted(value) if isinstance(value,set) else list(value)
    return description

def recipe_description(attack,name):
    budget=attack.goal_function.query_budget
    return dict(recipe=RECIPES[name].__name__,transformation=canonical_repr(attack.transformation),
        constraints=[describe_component(c) for c in attack.constraints],
        pre_transformation_constraints=[describe_component(c) for c in attack.pre_transformation_constraints],
        search=canonical_repr(attack.search_method),goal=type(attack.goal_function).__name__,
        query_budget='unlimited' if math.isinf(budget) else budget,
        goal_model_batch_size=attack.goal_function.batch_size,use_cache=attack.goal_function.use_cache,
        wrapper_forward_batch_size=1,is_black_box=attack.is_black_box,
        recipe_source_sha256=hashlib.sha256(inspect.getsource(RECIPES[name]).encode()).hexdigest(),
        goal_source_sha256=hashlib.sha256(inspect.getsource(type(attack.goal_function)).encode()).hexdigest(),
        search_source_sha256=digest_file(inspect.getsourcefile(type(attack.search_method))),
        goal_module_sha256=digest_file(inspect.getsourcefile(type(attack.goal_function))),
        constraint_source_sha256={type(c).__name__:digest_file(inspect.getsourcefile(type(c)))
            for c in [*attack.constraints,*attack.pre_transformation_constraints]})

def build_attack(name,wrapper,register=False):
    attack=RECIPES[name].build(wrapper)
    original=recipe_description(attack,name)
    assert isinstance(attack.goal_function,UntargetedClassification)
    assert attack.search_method.wir_method=='delete'
    names={type(c).__name__ for c in [*attack.constraints,*attack.pre_transformation_constraints]}
    required={'RepeatModification','StopwordModification','UniversalSentenceEncoder'}
    if name=='textfooler': required|={'WordEmbeddingDistance','PartOfSpeech','InputColumnModification'}
    assert required==names, (name,required,names)
    use=next(c for c in attack.constraints if type(c).__name__=='UniversalSentenceEncoder')
    assert math.isclose(float(use.threshold),0.840845057 if name=='textfooler' else 0.8,abs_tol=1e-9)
    if name=='textfooler':
        assert type(attack.transformation).__name__=='WordSwapEmbedding' and attack.transformation.max_candidates==50
        embedding_constraint=next(c for c in attack.constraints if type(c).__name__=='WordEmbeddingDistance')
        assert embedding_constraint.min_cos_sim==0.5
    else:
        assert type(attack.transformation).__name__=='CompositeTransformation'
        children=attack.transformation.transformations
        assert [type(t).__name__ for t in children]==['WordSwapRandomCharacterInsertion','WordSwapRandomCharacterDeletion',
            'WordSwapNeighboringCharacterSwap','WordSwapHomoglyphSwap','WordSwapEmbedding']
        assert children[-1].max_candidates==5
    assert attack.goal_function.use_cache and attack.goal_function.batch_size==32
    assert original['query_budget']=='unlimited', 'Unexpected library default; inspect before reproducing'
    if QUERY_CAP is not None: attack.goal_function.query_budget=QUERY_CAP
    description=dict(original=original,resolved=recipe_description(attack,name),
        query_cap_deviation=QUERY_CAP is not None,between_example_cache_policy='clear all; keep default within-example cache')
    if register: RECIPE_MANIFEST[name]=description
    elif name in RECIPE_MANIFEST: assert description==RECIPE_MANIFEST[name], 'Recipe changed between victims'
    return attack

class ProbeWrapper(ModelWrapper):
    def __init__(self): self.model=torch.nn.Linear(1,2)
    def __call__(self,texts): return np.tile(np.array([[0.9,0.1]],dtype=np.float32),(len(texts),1))

with stage('mandatory recipe resources and actual model forward preflight'):
    probe='This engaging movie has a thoughtful story with memorable characters and excellent performances throughout the entire film.'
    for name in ATTACK_NAMES:
        attack=build_attack(name,ProbeWrapper(),register=True)
        for c in attack.constraints:
            if type(c).__name__=='UniversalSentenceEncoder':
                embeddings=np.asarray(c.encode([probe,probe]))
                assert embeddings.shape[0]==2 and embeddings.ndim==2 and np.isfinite(embeddings).all()
            if type(c).__name__=='PartOfSpeech':
                assert c.tagger_type=='nltk'
                assert nltk.pos_tag(AttackedText(probe).words,tagset=c.tagset,lang=c.language_nltk)
        at=AttackedText(probe)
        candidates=attack.transformation(at)
        assert candidates, f'{name}: unavailable embedding/transformation resources'
        attack.filter_transformations(candidates[:8],at,original_text=at)
        attack.goal_function.query_budget=32 # synthetic preflight only; never used in scientific jobs
        result=attack.attack(probe,0)
        assert isinstance(result,FailedAttackResult), 'Synthetic recipe search did not complete normally'
        del attack; free_gpu()
    # Verify both independent initializations, official PuRe pca_lowrank on CUDA, wrapper/API compatibility.
    for model_id in MODEL_IDS:
        model=new_model(model_id).to(DEVICE)
        # A tiny backward probe verifies gradients through the unmodified PCR/PFSA before the grid.
        model.train(); probe_tokens=tokenizer([probe],truncation=True,max_length=MAX_LENGTH,return_tensors='pt')
        probe_tokens={k:v.to(DEVICE) for k,v in probe_tokens.items()}
        loss=model(**probe_tokens,labels=torch.tensor([0],device=DEVICE)).loss
        assert torch.isfinite(loss).item()
        loss.backward()
        assert model.bert.embeddings.word_embeddings.weight.grad is not None
        assert torch.isfinite(model.bert.embeddings.word_embeddings.weight.grad).all().item()
        model.zero_grad(set_to_none=True); del loss,probe_tokens
        wrapper=VictimWrapper(model,tokenizer)
        probabilities=wrapper([probe])
        assert probabilities.shape==(1,2) and np.isfinite(probabilities).all()
        assert np.all(probabilities>=0) and np.allclose(probabilities.sum(-1),1.0)
        with preserve_rng(), tempfile.TemporaryDirectory(prefix='checkpoint_preflight_',dir=ROOT) as temporary:
            initial_state=capture_rng(); saved_scores=wrapper([probe])
            model.save_pretrained(temporary,safe_serialization=True); tokenizer.save_pretrained(temporary)
            cls=BertForSequenceClassification if model_id=='bert_baseline' else PureBertForSequenceClassification
            reloaded,info=cls.from_pretrained(temporary,local_files_only=True,output_loading_info=True,
                **(PURE_KWARGS if model_id=='bert_pure' else {}))
            assert not any(info.get(k) for k in ('missing_keys','unexpected_keys','mismatched_keys','error_msgs')), info
            if model_id=='bert_pure': verify_pure_settings(reloaded)
            probe_wrapper=VictimWrapper(reloaded.to(DEVICE),AutoTokenizer.from_pretrained(temporary,local_files_only=True))
            restore_rng(initial_state)
            assert np.allclose(saved_scores,probe_wrapper([probe]),atol=1e-6), 'Preflight checkpoint round trip changed scores'
            del probe_wrapper,reloaded
        LOG.info('%s forward / save-reload / wrapper checks passed without optimization',model_id)
        for name in ATTACK_NAMES:
            attack=build_attack(name,wrapper)
            before=capture_rng(); direct=wrapper([probe])[0]; restore_rng(before)
            initial,_=attack.goal_function.init_attack_example(AttackedText(probe),int(direct.argmax()))
            assert np.allclose(np.asarray(initial.raw_output),direct,atol=1e-6), 'TextAttack initial probability mismatch'
            assert attack.goal_function.num_queries==1
            physical_before=wrapper.physical_queries
            cached,_=attack.goal_function.get_result(AttackedText(probe))
            assert attack.goal_function.num_queries==2, 'TextAttack logical queries must include cached requests'
            assert wrapper.physical_queries==physical_before, 'Pinned recipe model cache did not reuse the initial query'
            assert np.allclose(np.asarray(cached.raw_output),np.asarray(initial.raw_output))
            attack.clear_cache()
            # Real-model one-example API search (diagnostic, separately capped and not scientific output).
            attack.goal_function.query_budget=16
            result=attack.attack(probe,int(direct.argmax()))
            assert isinstance(result,(SuccessfulAttackResult,FailedAttackResult,SkippedAttackResult))
            del attack
        del wrapper,model; free_gpu()
    atomic_json(ROOT/'recipe_configuration.json',RECIPE_MANIFEST)
    PREFLIGHT_OK=True
    LOG.info('All mandatory resources, both recipes, and both model forwards passed')


2026-10-31 18:00:44,351 INFO Starting mandatory recipe resources and actual model forward preflight
textattack: Downloading https://textattack.s3.amazonaws.com/word_embeddings/paragramcf.
100%|██████████| 481M/481M [00:07<00:00, 68.7MB/s]
textattack: Unzipping file /kaggle/working/.pure_cache/textattack/tmpvfab77tr.zip to /kaggle/working/.pure_cache/textattack/word_embeddings/paragramcf.
textattack: Successfully saved word_embeddings/paragramcf to cache.
textattack: Unknown if model of class <class 'torch.nn.modules.linear.Linear'> compatible with goal function <class 'textattack.goal_functions.classification.untargeted_classification.UntargetedClassification'>.
2026-10-31 18:01:02,484 INFO Using /kaggle/working/.pure_cache/textattack to cache modules.
2026-10-31 18:01:02,487 INFO Downloading TF-Hub Module 'https://tfhub.dev/google/universal-sentence-encoder/3'.
2026-10-31 18:01:12,846 INFO Downloaded https://tfhub.dev/google/universal-sentence-encoder/3, Total size: 987.45MB
2026-10-3

## 8. Fingerprints and restoration of verified state

The experiment fingerprint includes seed, all scientific settings, environment identity, immutable data/model revisions, split content hashes, official source SHA256, pretrained weight SHA256 and recipe descriptions. Runtime stage/job selection and wall times do not change scientific identity. Attack jobs additionally bind both winning checkpoint hashes and the frozen clean predictions. Thus a new model or clean run cannot silently inherit attacks from a different victim.


In [8]:
IDENTITY=dict(request=REQUEST,environment=ENV_IDENTITY,dataset_revision=DATASET_COMMIT,
    dataset_splits=split_info,model_revision=MODEL_COMMIT,pretrained_sha256=PRETRAINED_HASH,
    initial_shared_backbone_sha256=INITIAL_SHARED_HASH,source_sha256=SOURCE_HASHES,recipes=RECIPE_MANIFEST)
EXPERIMENT_FP=stable_hash(IDENTITY)
EXPERIMENT_CONFIG=dict(request=REQUEST,identity=IDENTITY,experiment_fingerprint=EXPERIMENT_FP,
    dataset_revision=DATASET_COMMIT,model_revision=MODEL_COMMIT,pure_commit=PURE_COMMIT,
    source_sha256=SOURCE_HASHES,implementation_defaults=IMPLEMENTATION_DEFAULTS,
    paper_specified=['4 epochs','AdamW','learning-rate and batch-size grid','default TextAttack recipes','entire test set'],
    implementation_choices=TRAIN_SETTINGS,seed_note='42 is an implementation choice, not a claimed author seed',
    protocol_approximation=HYPERPARAM_MODE=='fixed',
    status='SETUP_COMPLETE_NO_MEASURED_TEST_RESULTS_YET')
if PRIOR_CONFIG:
    assert PRIOR_CONFIG['experiment_fingerprint']==EXPERIMENT_FP, 'Prior source/data/environment identity differs; refuse resume'
    if PRIOR_ROOT != ROOT:
        # Copy only after matching identity. Read-only mounted artifacts remain unchanged.
        for dirname in ('models','candidates','clean_records','attack_records'):
            source=PRIOR_ROOT/dirname
            if source.exists():
                for p in source.rglob('*'):
                    if not p.is_file() or any(part.endswith('.tmp') for part in p.relative_to(PRIOR_ROOT).parts): continue
                    target=ROOT/p.relative_to(PRIOR_ROOT); target.parent.mkdir(parents=True,exist_ok=True)
                    if target.exists(): assert digest_file(target)==digest_file(p), f'Conflicting artifact: {target}'
                    else: shutil.copy2(p,target)
        for name in ('training_summary.json','training_history.csv','clean_test_metrics.json',
                     'clean_test_predictions.csv','inference_diagnostics.json'):
            p=PRIOR_ROOT/name
            if p.exists():
                target=ROOT/name
                if target.exists(): assert digest_file(target)==digest_file(p), f'Conflicting report: {target}'
                else: shutil.copy2(p,target)
EXPERIMENT_CONFIG['review_patch']='persistence-100; resume accounting; mismatch diagnostics; Kaggle preflight (scientific identity unchanged)'
atomic_json(ROOT/'experiment_config.json',EXPERIMENT_CONFIG)


## 9. Four-epoch training grid and safe checkpoint selection

Only `train` enters optimization; only `validation` selects an epoch/configuration. TEST never appears in a Trainer dataset. Each candidate starts independently from untouched pretrained BERT, including PuRe. Validation runs every epoch; no early-stopping callback is installed. Trainer uses maintained `eval_strategy`, `processing_class` and PyTorch AdamW APIs.

A checkpoint becomes resumable only after an `on_save` manifest hashes weights, optimizer, scheduler, RNG and Trainer state. Resume chooses the latest intact committed checkpoint and rejects corrupted committed state. An incomplete save without a commit marker is ignored and logged. Training loss is sample weighted over examples observed in each callback window; observed/expected counts and whether a window is partial are logged. A resumed window need not contain exactly 6,920 newly observed examples. Dataset sizes stay strict. Completion checks Trainer's saved progress and four completed validation epochs, rather than asserting a hand-calculated optimizer-step total or exact floating-point epoch equality. Live logs survive interruption. Candidate completion requires four epochs even when the best validation epoch is earlier. After saving the hash-verified candidate winner, bulky optimizer checkpoints are removed to control Kaggle disk use. Candidate best weights remain to verify completed work without retraining. Winners are copied to `models/<model_id>/best_model/` with tokenizer, training history, hyperparameters and provenance.


In [9]:
def files_manifest(directory):
    directory=Path(directory)
    return {str(p.relative_to(directory)):digest_file(p) for p in sorted(directory.rglob('*'))
            if p.is_file() and p.name not in {'artifact_manifest.json','checkpoint_commit.json'} and not p.name.endswith('.tmp')}

def verify_manifest(directory,manifest):
    directory=Path(directory)
    assert manifest, f'Empty artifact manifest: {directory}'
    for relative,sha in manifest.items():
        p=directory/relative
        assert p.is_file() and digest_file(p)==sha, f'Missing/corrupt artifact: {p}'
    assert files_manifest(directory)==manifest, f'Untracked or modified artifact files: {directory}'

def verify_model_directory(directory):
    directory=Path(directory)
    record=unseal(read_json(directory/'artifact_manifest.json'))
    assert record['experiment_fingerprint']==EXPERIMENT_FP
    verify_manifest(directory,record['files'])
    assert (directory/'model.safetensors').is_file() and (directory/'config.json').is_file()
    assert (directory/'tokenizer_config.json').is_file()
    return record

class EpochLossTrainer(Trainer):
    def __init__(self,*args,**kwargs):
        super().__init__(*args,**kwargs)
        self.epoch_loss_sum=0.0; self.epoch_loss_samples=0
        self.model_accepts_loss_kwargs=False
    def compute_loss(self,model,inputs,return_outputs=False,num_items_in_batch=None):
        count=int(inputs['labels'].shape[0])
        output=super().compute_loss(model,inputs,return_outputs=return_outputs,num_items_in_batch=num_items_in_batch)
        loss=output[0] if return_outputs else output
        if model.training:
            self.epoch_loss_sum+=float(loss.detach().item())*count
            self.epoch_loss_samples+=count
        return output

class AuditTrainingCallback(TrainerCallback):
    def __init__(self,candidate_fp):
        self.candidate_fp=candidate_fp; self.trainer=None
    def on_epoch_end(self,args,state,control,**kwargs):
        trainer=self.trainer
        assert trainer is not None
        observed=trainer.epoch_loss_samples; expected=len(trainer.train_dataset)
        if observed != expected:
            LOG.warning('Epoch callback window: observed=%d expected=%d epoch=%s step=%d; resumed windows can be partial',
                        observed,expected,state.epoch,state.global_step)
        window={'train_epoch_samples':observed,'train_epoch_expected_samples':expected,
                'train_epoch_window_partial':observed!=expected}
        if observed:
            window['train_epoch_loss']=trainer.epoch_loss_sum/observed
        trainer.log(window)
        trainer.epoch_loss_sum=0.0; trainer.epoch_loss_samples=0
    def on_log(self,args,state,control,logs=None,**kwargs):
        for k,v in (logs or {}).items():
            if isinstance(v,(int,float)) and not math.isfinite(v): raise RuntimeError(f'Nonfinite {k}: {v}')
        atomic_json(Path(args.output_dir).parent/'live_training_history.json',state.log_history)
    def on_save(self,args,state,control,**kwargs):
        directory=Path(args.output_dir)/f'checkpoint-{state.global_step}'
        for name in ('trainer_state.json','optimizer.pt','scheduler.pt','rng_state.pth'):
            assert (directory/name).is_file(), f'Incomplete resumable checkpoint: {directory/name}'
        saved_state=read_json(directory/'trainer_state.json')
        assert saved_state['global_step']==int(state.global_step), 'Callback ran before matching Trainer state was saved'
        assert (directory/'model.safetensors').is_file(), 'Model weights missing at on_save'
        atomic_json(directory/'checkpoint_commit.json',seal(dict(candidate_fingerprint=self.candidate_fp,
            step=int(state.global_step),files=files_manifest(directory))))
        LOG.info('Committed Trainer checkpoint %s (epoch=%s, optimizer step=%d)',directory,state.epoch,state.global_step)

def latest_committed_checkpoint(directory,candidate_fp):
    choices=[]
    for p in Path(directory).glob('checkpoint-*'):
        commit=p/'checkpoint_commit.json'
        if not commit.exists():
            LOG.warning('Ignoring interrupted, uncommitted checkpoint %s',p); continue
        record=unseal(read_json(commit)); assert record['candidate_fingerprint']==candidate_fp
        verify_manifest(p,record['files']); choices.append((record['step'],p))
    return str(max(choices)[1]) if choices else None

def completed_validation_history(history):
    by_step={}
    for entry in history:
        if 'eval_accuracy' not in entry: continue
        row=dict(entry); step=int(row['step'])
        if step in by_step:
            assert by_step[step]==row, 'Conflicting validation logs for the same optimizer step'
        by_step[step]=row
    records=[by_step[k] for k in sorted(by_step)]
    completed={int(round(float(r['epoch']))) for r in records
               if math.isclose(float(r['epoch']),round(float(r['epoch'])),abs_tol=1e-3)}
    assert set(range(1,EPOCHS+1))<=completed, 'Missing completed validation epoch(s); refuse to mark four-epoch training complete'
    return records

def compute_validation_metrics(prediction):
    logits,labels=prediction
    if isinstance(logits,tuple): logits=logits[0]
    assert len(labels)==EXPECTED_SIZES['validation'] and np.isfinite(logits).all()
    return {'accuracy':float((np.asarray(logits).argmax(-1)==np.asarray(labels)).mean())}

def tokenized_split(split):
    # No text or sample identifiers are passed to the model; originals remain in SPLIT_ROWS.
    ds=raw_data[split].map(lambda b: tokenizer(b['text'],truncation=True,max_length=MAX_LENGTH),batched=True)
    return ds.remove_columns([c for c in ds.column_names if c not in {'input_ids','attention_mask','token_type_ids','label'}])

TRAIN_DS,VALIDATION_DS=tokenized_split('train'),tokenized_split('validation')
COLLATOR=DataCollatorWithPadding(tokenizer=tokenizer,padding=True,return_tensors='pt')
GRID=[(lr,b) for lr in [1e-5,2e-5,5e-5] for b in [8,16,32]] if HYPERPARAM_MODE=='paper_grid' else [(FIXED_LR,FIXED_BATCH)]
assert len(GRID)==(9 if HYPERPARAM_MODE=='paper_grid' else 1)

# Save-pretrained on official PuRe does not automatically preserve custom constructor arguments.
# Store those in both config and provenance, and pass them explicitly at every reload.
def export_model(model,destination,provenance,history):
    destination=Path(destination); temporary=destination.with_name(destination.name+'.tmp')
    if temporary.exists(): shutil.rmtree(temporary)
    temporary.mkdir(parents=True,exist_ok=True)
    model.config.reproduction_pure_kwargs=PURE_KWARGS if provenance['model']=='bert_pure' else None
    model.config.reproduction_experiment_fingerprint=EXPERIMENT_FP
    model.save_pretrained(temporary,safe_serialization=True)
    tokenizer.save_pretrained(temporary)
    atomic_json(temporary/'hyperparameters.json',provenance)
    atomic_json(temporary/'training_history.json',history)
    atomic_json(temporary/'provenance.json',dict(provenance,source_sha256=SOURCE_HASHES,
        pretrained_commit=MODEL_COMMIT,pretrained_sha256=PRETRAINED_HASH,initial_shared_backbone_sha256=INITIAL_SHARED_HASH))
    files=files_manifest(temporary)
    atomic_json(temporary/'artifact_manifest.json',seal(dict(experiment_fingerprint=EXPERIMENT_FP,files=files)))
    if destination.exists():
        # An existing committed artifact must be used, not overwritten.
        verify_model_directory(destination)
        raise RuntimeError(f'Committed model already exists without expected completion state: {destination}')
    os.replace(temporary,destination)
    return verify_model_directory(destination)

def train_candidate(model_id,lr,batch):
    assert PREFLIGHT_OK
    micro=min(MAX_MICROBATCH,batch); assert batch%micro==0
    cid=f'lr_{lr:.0e}_bs_{batch}'
    directory=ROOT/'candidates'/model_id/cid; directory.mkdir(parents=True,exist_ok=True)
    spec=dict(experiment_fingerprint=EXPERIMENT_FP,model=model_id,lr=lr,batch_size=batch,
              microbatch=micro,gradient_accumulation_steps=batch//micro,seed=SEED,settings=TRAIN_SETTINGS)
    candidate_fp=stable_hash(spec)
    marker=directory/'completed.json'
    if not marker.exists() and (directory/'best_model').exists():
        manifest=verify_model_directory(directory/'best_model')
        provenance=read_json(directory/'best_model'/'hyperparameters.json')
        history=read_json(directory/'best_model'/'training_history.json')
        assert provenance['candidate_fingerprint']==candidate_fp and provenance['trained_epochs']==4
        epoch_history=completed_validation_history(history)
        recovered=dict(provenance,best_model_manifest=manifest,history=history,epoch_history=epoch_history,
                       training_result={'recovered_from_atomic_complete_model_export':True})
        atomic_json(marker,seal(recovered))
    if marker.exists():
        completed=unseal(read_json(marker)); assert completed['candidate_fingerprint']==candidate_fp
        verified=verify_model_directory(directory/'best_model')
        assert completed['best_model_manifest']==verified
        completed_validation_history(completed['history'])
        assert completed['trained_epochs']==4
        LOG.info('Skipping hash-verified completed %s / %s',model_id,cid)
        if (directory/'checkpoints').exists(): shutil.rmtree(directory/'checkpoints')
        return completed
    free_bytes=shutil.disk_usage(ROOT).free
    if free_bytes < 6*2**30:
        raise RuntimeError('Less than 6 GiB free: persist artifacts and free cache/output storage before resuming. No hyperparameter was changed or completed candidate retrained.')
    resumed=latest_committed_checkpoint(directory/'checkpoints',candidate_fp)
    model=trainer=None
    t=time.perf_counter()
    atomic_json(directory/'candidate_config.json',spec)
    try:
        model=new_model(model_id)
        args=TrainingArguments(output_dir=str(directory/'checkpoints'),num_train_epochs=4,
            per_device_train_batch_size=micro,gradient_accumulation_steps=batch//micro,
            per_device_eval_batch_size=1,learning_rate=lr,weight_decay=0.01,
            adam_beta1=0.9,adam_beta2=0.999,adam_epsilon=1e-8,optim='adamw_torch',
            lr_scheduler_type='linear',warmup_ratio=0.0,max_grad_norm=1.0,
            eval_strategy='epoch',save_strategy='epoch',logging_strategy='steps',logging_steps=25,
            load_best_model_at_end=True,metric_for_best_model='accuracy',greater_is_better=True,
            save_total_limit=2,seed=SEED,data_seed=SEED,fp16=False,bf16=False,tf32=False,
            gradient_checkpointing=True,gradient_checkpointing_kwargs={'use_reentrant':False},
            label_smoothing_factor=0.0,report_to='none',dataloader_num_workers=0,
            dataloader_pin_memory=True,save_safetensors=True,save_only_model=False)
        audit_callback=AuditTrainingCallback(candidate_fp)
        trainer=EpochLossTrainer(model=model,args=args,train_dataset=TRAIN_DS,eval_dataset=VALIDATION_DS,
            processing_class=tokenizer,data_collator=COLLATOR,compute_metrics=compute_validation_metrics,
            callbacks=[audit_callback])
        audit_callback.trainer=trainer
        LOG.info('Training %s %s: micro=%s accumulation=%s resume=%s',model_id,cid,micro,batch//micro,resumed)
        result=trainer.train(resume_from_checkpoint=resumed)
        assert int(trainer.state.max_steps)>0 and trainer.state.global_step>=trainer.state.max_steps, 'Trainer stopped before its scheduled four-epoch run completed'
        epoch_history=completed_validation_history(trainer.state.log_history)
        train_epochs=[x for x in trainer.state.log_history if 'train_epoch_samples' in x]
        LOG.info('Trainer completed: reported epoch=%s optimizer steps=%d scheduled=%d observed sample windows=%s',
                 trainer.state.epoch,trainer.state.global_step,trainer.state.max_steps,
                 [x['train_epoch_samples'] for x in train_epochs])
        selected=min(epoch_history,key=lambda x:(-x['eval_accuracy'],x['epoch']))
        expected_best=f'checkpoint-{selected["step"]}'
        assert Path(trainer.state.best_model_checkpoint).name==expected_best, 'Unexpected checkpoint tie-breaking'
        assert math.isclose(trainer.state.best_metric,selected['eval_accuracy'],abs_tol=1e-12)
        # Record actual/scheduled steps. Do not require an independently rounded fixed step count.
        assert trainer.args.num_train_epochs==EPOCHS
        assert trainer.args.per_device_train_batch_size*trainer.args.gradient_accumulation_steps==batch
        selected_directory=Path(trainer.state.best_model_checkpoint)
        commit=unseal(read_json(selected_directory/'checkpoint_commit.json'))
        assert commit['candidate_fingerprint']==candidate_fp
        verify_manifest(selected_directory,commit['files'])
        previous_time=read_json(directory/'timing.json').get('completed_session_seconds',0.0) if (directory/'timing.json').exists() else 0.0
        elapsed=time.perf_counter()-t
        provenance=dict(spec,candidate_id=cid,candidate_fingerprint=candidate_fp,
            selected_epoch=int(round(selected['epoch'])),selected_checkpoint=expected_best,
            validation_accuracy=float(selected['eval_accuracy']),validation_loss=float(selected['eval_loss']),
            trained_epochs=4,optimizer_steps=int(trainer.state.global_step),
            trainer_scheduled_optimizer_steps=int(trainer.state.max_steps),trainer_reported_epoch=trainer.state.epoch,
            train_epoch_sample_windows=[{k:v for k,v in x.items() if k.startswith('train_epoch_') or k in {'epoch','step'}} for x in train_epochs],
            completed_session_wall_seconds=previous_time+elapsed,interrupted_time_may_be_unobserved=True,
            dropout=dict(hidden=model.config.hidden_dropout_prob,attention=model.config.attention_probs_dropout_prob,
                         classifier=model.dropout.p),max_length=MAX_LENGTH,
            attention_implementation=getattr(model.config,'_attn_implementation',None),
            scheduler='linear / zero warmup',weight_decay=0.01,seed_origin='implementation choice',
            physical_batch_deviation='microbatch<=8 with gradient accumulation; FP32 + gradient checkpointing')
        manifest=export_model(trainer.model,directory/'best_model',provenance,trainer.state.log_history)
        completed=dict(provenance,best_model_manifest=manifest,epoch_history=epoch_history,
            history=trainer.state.log_history,training_result=result.metrics)
        atomic_json(marker,seal(completed))
        # Cleanup only after a completed, verified export; keep lightweight records and candidate best weights.
        shutil.rmtree(directory/'checkpoints')
        return completed
    except torch.cuda.OutOfMemoryError as exc:
        atomic_json(directory/'oom.json',dict(error=str(exc),candidate=spec,
            action='aborted; no automatic hyperparameter/precision change; inspect T4 memory and resume'))
        raise RuntimeError('CUDA OOM: candidate state retained; no silent batch reduction') from exc
    finally:
        elapsed=time.perf_counter()-t
        timing=read_json(directory/'timing.json') if (directory/'timing.json').exists() else {'completed_session_seconds':0.0}
        timing['completed_session_seconds']+=elapsed
        atomic_json(directory/'timing.json',timing)
        if trainer is not None:
            for callback in trainer.callback_handler.callbacks:
                if isinstance(callback,AuditTrainingCallback): callback.trainer=None
        del trainer,model; free_gpu()

def refresh_training_reports():
    histories=[]; selections={}; completed_counts={}; candidate_summaries=[]
    for model_id in MODEL_IDS:
        completed=[]
        for lr,batch in GRID:
            cid=f'lr_{lr:.0e}_bs_{batch}'
            directory=ROOT/'candidates'/model_id/cid
            marker=directory/'completed.json'
            if marker.exists():
                record=unseal(read_json(marker)); assert record['experiment_fingerprint']==EXPERIMENT_FP
                completed.append(record); history=record['history']
                candidate_summaries.append({k:v for k,v in record.items() if k not in {'history','epoch_history','best_model_manifest','training_result'}})
            elif (directory/'live_training_history.json').exists():
                history=read_json(directory/'live_training_history.json')
            else: history=[]
            for row in history:
                histories.append(dict(model=model_id,candidate_id=cid,lr=lr,batch_size=batch,**row))
        completed_counts[model_id]=len(completed)
        if len(completed)==len(GRID):
            best=min(completed,key=lambda r:(-r['validation_accuracy'],r['selected_epoch'],r['lr'],r['batch_size']))
            selections[model_id]={k:v for k,v in best.items() if k not in {'history','epoch_history','best_model_manifest'}}
            selections[model_id]['winner_exported']=(ROOT/'models'/model_id/'best_model'/'artifact_manifest.json').exists()
    atomic_csv(ROOT/'training_history.csv',pd.DataFrame(histories,columns=list(dict.fromkeys([
        'model','candidate_id','lr','batch_size',*[k for r in histories for k in r]]))))
    atomic_json(ROOT/'training_summary.json',dict(experiment_fingerprint=EXPERIMENT_FP,
        status='complete' if len(selections)==2 and all(r['winner_exported'] for r in selections.values()) else 'INCOMPLETE',
        completed_candidate_counts=completed_counts,expected_candidates_per_model=len(GRID),
        candidates=candidate_summaries,selections=selections))

def train_models():
    histories=[]; selections={}; candidate_summaries=[]
    for model_id in MODEL_IDS:
        candidates=[train_candidate(model_id,lr,batch) for lr,batch in GRID]
        assert len(candidates)==len(GRID)
        candidate_summaries.extend([{k:v for k,v in r.items() if k not in {'history','epoch_history','best_model_manifest','training_result'}} for r in candidates])
        best=min(candidates,key=lambda r:(-r['validation_accuracy'],r['selected_epoch'],r['lr'],r['batch_size']))
        source=ROOT/'candidates'/model_id/best['candidate_id']/'best_model'
        target=ROOT/'models'/model_id/'best_model'
        verify_model_directory(source)
        if target.exists():
            assert verify_model_directory(target)==verify_model_directory(source), 'Winner changed unexpectedly'
        else:
            target.parent.mkdir(parents=True,exist_ok=True)
            temp=target.with_name('best_model.tmp')
            if temp.exists(): shutil.rmtree(temp)
            shutil.copytree(source,temp); os.replace(temp,target)
        verify_model_directory(target)
        selections[model_id]={k:v for k,v in best.items() if k not in {'history','epoch_history','best_model_manifest'}}
        selections[model_id]['winner_directory']=str(target)
        selections[model_id]['winner_manifest_sha256']=stable_hash(verify_model_directory(target))
        for candidate in candidates:
            for row in candidate['history']:
                histories.append(dict(model=model_id,candidate_id=candidate['candidate_id'],
                    lr=candidate['lr'],batch_size=candidate['batch_size'],**row))
        # Update after each model so a later interruption preserves a useful training report.
        atomic_json(ROOT/'training_summary.json',dict(experiment_fingerprint=EXPERIMENT_FP,
            status='complete' if len(selections)==2 else 'INCOMPLETE',selections=selections,candidates=candidate_summaries))
        atomic_csv(ROOT/'training_history.csv',pd.DataFrame(histories))
    return selections


Map:   0%|          | 0/6920 [00:00<?, ? examples/s]

Map:   0%|          | 0/872 [00:00<?, ? examples/s]

## 10. Reload winners, complete clean TEST, and inference diagnostics

Reload model weights and tokenizer from each saved winner and reject any missing/unexpected tensors. PuRe constructor settings are explicitly restored, since they are not standard BertConfig constructor fields. The checkpoint must retain the exact source and experiment identity. A same-RNG save/reload forward comparison runs before TEST evaluation.

Stage-level seeds use fixed documented offsets: clean `SEED+1000+model_index`, diagnostics `SEED+3000`, attacks `SEED+5000+job_index`. They initialize whole stage/job streams, never individual texts. Training candidates use `SEED`. Clean inference uses all 1,821 samples per model, in original order, before any scientific attacks. Per-sample clean records contain post-inference RNG states for resume, so restarting does not reseed each text. Metrics include cross-entropy, positive-class precision/recall/F1, macro F1 and a labeled confusion matrix. Validation accuracy remains separately labeled in training reports.

Diagnostics use 64 fixed test indices (not an attack subset) and five repeated passes: stochastic repeated singleton forwards; singleton versus padded batches; and a same-RNG singleton-versus-fixed-padding comparison to isolate padding effects. Diagnostic forwards preserve the surrounding RNG state and never alter frozen eligibility. Actual TextAttack initial scores are compared with direct scores under the same restored RNG state. Across-time stochastic disagreements are reported rather than hidden.

`paper_like_stochastic` leaves randomized SVD active and does not reseed modified texts. `deterministic_diagnostic` requests strict deterministic PyTorch kernels and writes to a separate output lineage. It replays whole ordered RNG streams; it never seeds an original or modified text individually. Randomized SVD still consumes random draws, so this does not create a stateless deterministic classifier. Controlled same-RNG replay is a sensitivity diagnostic, never presented as the paper protocol. Neither policy patches pooling or SVD.


In [10]:
def winner_directory(model_id):
    return ROOT/'models'/model_id/'best_model'

def load_victim(model_id):
    directory=winner_directory(model_id); manifest=verify_model_directory(directory)
    provenance=read_json(directory/'provenance.json')
    assert provenance['model']==model_id and provenance['experiment_fingerprint']==EXPERIMENT_FP
    assert provenance['trained_epochs']==4 and provenance['source_sha256']==SOURCE_HASHES
    cls=BertForSequenceClassification if model_id=='bert_baseline' else PureBertForSequenceClassification
    config=AutoConfig.from_pretrained(directory,local_files_only=True)
    assert config.reproduction_experiment_fingerprint==EXPERIMENT_FP
    kwargs={}
    if model_id=='bert_pure':
        assert config.reproduction_pure_kwargs==PURE_KWARGS
        kwargs=PURE_KWARGS
    model,info=cls.from_pretrained(directory,config=config,local_files_only=True,
        output_loading_info=True,**kwargs)
    assert not any(info.get(k) for k in ('missing_keys','unexpected_keys','mismatched_keys','error_msgs')), info
    if model_id=='bert_pure': verify_pure_settings(model)
    tok=AutoTokenizer.from_pretrained(directory,local_files_only=True)
    assert tok.get_vocab()==tokenizer.get_vocab() and tok.padding_side=='right'
    model.to(DEVICE).eval()
    return VictimWrapper(model,tok)

def check_checkpoint_reload(model_id):
    # Round trip a single loaded winner; no fine-tuning and no test selection.
    wrapper=load_victim(model_id)
    probe='A checkpoint round trip preserves the trained classifier.'
    with preserve_rng():
        before=capture_rng(); first=wrapper([probe])[0]
        del wrapper; free_gpu()
        wrapper=load_victim(model_id); restore_rng(before)
        second=wrapper([probe])[0]
        assert np.allclose(first,second,atol=1e-6), f'Checkpoint reload predictions changed: {model_id}'
    del wrapper; free_gpu()
    return True

def classification_metrics(rows):
    assert len(rows)==1821 and len({r['sample_id'] for r in rows})==1821
    labels=[r['label'] for r in rows]; predictions=[r['prediction'] for r in rows]
    precision,recall,f1,_=precision_recall_fscore_support(labels,predictions,average='binary',zero_division=0)
    cm=confusion_matrix(labels,predictions,labels=[0,1]).tolist()
    correct=sum(r['correct'] for r in rows)
    return dict(split='TEST',N=1821,N_correct=correct,accuracy_pct=pct(correct,1821),
        cross_entropy_loss=float(np.mean([r['loss'] for r in rows])),precision_pct=100*float(precision),
        recall_pct=100*float(recall),f1_pct=100*float(f1),macro_f1_pct=100*float(f1_score(labels,predictions,average='macro')),
        confusion_matrix=cm,confusion_matrix_order=['negative','positive'],
        confusion_matrix_axes='rows=gold; columns=predicted',accuracy_numerator=correct,accuracy_denominator=1821,
        precision_numerator=cm[1][1],precision_denominator=cm[1][1]+cm[0][1],
        recall_numerator=cm[1][1],recall_denominator=cm[1][1]+cm[1][0],
        positive_f1_numerator=2*cm[1][1],positive_f1_denominator=2*cm[1][1]+cm[0][1]+cm[1][0],
        macro_f1_definition='unweighted mean of F1 for negative and positive; zero_division=0',macro_f1_class_count=2)

def clean_prefix(model_id):
    return stable_hash(dict(experiment=EXPERIMENT_FP,winner=verify_model_directory(winner_directory(model_id)),
                            model=model_id,inference=REQUEST['inference'],policy=RNG_POLICY))

def clean_model(model_id):
    prefix=clean_prefix(model_id); directory=ROOT/'clean_records'/model_id; directory.mkdir(parents=True,exist_ok=True)
    records=[]
    for p in sorted(directory.glob('example_*.json')):
        r=unseal(read_json(p)); assert r['clean_fingerprint']==prefix
        records.append(r)
    # Clean evaluation is an ordered prefix, so exact sequential RNG continuation is possible.
    assert len(records)<=1821
    assert [r['sample_id'] for r in records]==[r['sample_id'] for r in TEST_ROWS[:len(records)]], 'Clean ledger is not a valid prefix'
    for r,s in zip(records,TEST_ROWS):
        assert r['model']==model_id and r['label']==s['label'] and r['original_text']==s['text']
        assert r['prediction'] in (0,1) and r['correct']==(r['prediction']==r['label'])
    wrapper=load_victim(model_id)
    try:
        if records: restore_rng(records[-1]['rng_after'])
        else: set_seed(SEED+1000+MODEL_IDS.index(model_id))
        for index in tqdm(range(len(records),1821),desc=f'Complete clean TEST: {model_id}'):
            sample=TEST_ROWS[index]; p=wrapper([sample['text']])[0]
            pred=int(p.argmax()); label=sample['label']
            logits=wrapper.last_logits.astype(np.float64)
            row=dict(clean_fingerprint=prefix,model=model_id,seed=SEED,sample_id=sample['sample_id'],
                original_text=sample['text'],original_text_sha256=sample['original_text_sha256'],
                label=label,prediction=pred,correct=bool(pred==label),negative_probability=float(p[0]),
                positive_probability=float(p[1]),negative_logit=float(logits[0]),positive_logit=float(logits[1]),
                loss=float(np.logaddexp.reduce(logits)-logits[label]),rng_after=capture_rng())
            atomic_json(directory/f'example_{index:05d}.json',seal(row)); records.append(row)
        assert len(records)==1821
        metrics=classification_metrics(records)
        atomic_json(directory/'complete.json',seal(dict(clean_fingerprint=prefix,N=1821,
            predictions_sha256=stable_hash([{k:v for k,v in r.items() if k!='rng_after'} for r in records]),metrics=metrics)))
    finally:
        del wrapper; free_gpu()
    return records,metrics

def inference_diagnostics(model_id):
    wrapper=load_victim(model_id); result={}
    try:
        with preserve_rng():
            set_seed(SEED+3000)
            indices=np.linspace(0,1820,DIAGNOSTIC_N,dtype=int).tolist()
            texts=[TEST_ROWS[i]['text'] for i in indices]
            panel_start=capture_rng()
            repeated=[wrapper(texts).argmax(-1).tolist() for _ in range(DIAGNOSTIC_REPEATS)]
            panel_end=capture_rng()
            restore_rng(panel_start)
            replay=wrapper(texts).argmax(-1).tolist()
            replay_disagreements=sum(a!=b for a,b in zip(repeated[0],replay))
            assert replay_disagreements==0, 'Whole-stream replay under identical state changed predictions'
            restore_rng(panel_end)
            singleton=np.array(repeated[0]); batched=[]
            # Actual dynamic padding, with its own continuing rSVD draws.
            for i in range(0,len(texts),8):
                tokens=wrapper.tokenizer(texts[i:i+8],truncation=True,max_length=MAX_LENGTH,padding=True,return_tensors='pt')
                with torch.inference_mode():
                    logits=wrapper.model(**{k:v.to(DEVICE) for k,v in tokens.items()}).logits
                assert torch.isfinite(logits).all()
                batched.extend(logits.argmax(-1).cpu().tolist())
            isolated=[]; probability_deltas=[]
            for text in texts:
                state=capture_rng()
                a=wrapper([text])[0]
                restore_rng(state)
                tokens=wrapper.tokenizer(text,truncation=True,max_length=MAX_LENGTH,padding='max_length',return_tensors='pt')
                with torch.inference_mode():
                    b=wrapper.model(**{k:v.to(DEVICE) for k,v in tokens.items()}).logits.softmax(-1)[0].cpu().numpy()
                isolated.append(int(a.argmax())!=int(b.argmax()))
                probability_deltas.append(float(np.max(np.abs(a-b))))
            initial_consistency={}
            for name in ATTACK_NAMES:
                attack=build_attack(name,wrapper)
                errors=0
                for text in texts[:8]:
                    attack.clear_cache(); state=capture_rng(); direct=wrapper([text])[0]; restore_rng(state)
                    original,_=attack.goal_function.init_attack_example(AttackedText(text),int(direct.argmax()))
                    errors+=int(not np.allclose(np.asarray(original.raw_output),direct,atol=1e-6))
                assert errors==0, 'Wrapper and TextAttack disagree under identical RNG state'
                initial_consistency[name]=dict(N=8,disagreements=errors)
                del attack
            repeat_disagreements=sum(int(a!=b) for row in repeated[1:] for a,b in zip(repeated[0],row))
            result=dict(model=model_id,policy=RNG_POLICY,sample_ids=[TEST_ROWS[i]['sample_id'] for i in indices],
                N=DIAGNOSTIC_N,repeats=DIAGNOSTIC_REPEATS,whole_panel_same_rng_replay_disagreements=replay_disagreements,
                repeated_disagreements=repeat_disagreements,
                repeated_comparisons=DIAGNOSTIC_N*(DIAGNOSTIC_REPEATS-1),
                repeated_disagreement_pct=pct(repeat_disagreements,DIAGNOSTIC_N*(DIAGNOSTIC_REPEATS-1)),
                singleton_vs_dynamic_batch_disagreement_pct=pct(int(np.sum(singleton!=np.array(batched))),DIAGNOSTIC_N),
                singleton_vs_fixed_padding_same_rng_disagreement_pct=pct(sum(isolated),DIAGNOSTIC_N),
                fixed_padding_mean_max_probability_delta=float(np.mean(probability_deltas)),
                textattack_same_rng_initial_consistency=initial_consistency,
                interpretation='Dynamic-batch comparison combines RNG and padding effects; fixed-padding check shares initial RNG. Official implementation is unchanged.')
    finally:
        del wrapper; free_gpu()
    return result

def load_complete_clean():
    clean={}; metrics={}
    for model_id in MODEL_IDS:
        directory=ROOT/'clean_records'/model_id; prefix=clean_prefix(model_id)
        completion=unseal(read_json(directory/'complete.json')); assert completion['clean_fingerprint']==prefix
        records=[unseal(read_json(directory/f'example_{i:05d}.json')) for i in range(1821)]
        assert [r['sample_id'] for r in records]==[r['sample_id'] for r in TEST_ROWS]
        assert all(r['clean_fingerprint']==prefix and r['model']==model_id for r in records)
        for r,s in zip(records,TEST_ROWS):
            assert r['original_text']==s['text'] and r['label']==s['label'] and r['correct']==(r['prediction']==s['label'])
        assert completion['predictions_sha256']==stable_hash([{k:v for k,v in r.items() if k!='rng_after'} for r in records])
        metrics[model_id]=classification_metrics(records); assert metrics[model_id]==completion['metrics']
        clean[model_id]={r['sample_id']:r for r in records}
    return clean,metrics

def evaluate_models():
    diagnostics={}
    for model_id in MODEL_IDS:
        assert check_checkpoint_reload(model_id)
        clean_model(model_id)
        diagnostics[model_id]=inference_diagnostics(model_id)
    clean,metrics=load_complete_clean()
    frame=pd.DataFrame([{k:v for k,v in r.items() if k not in {'rng_after','clean_fingerprint'}}
                        for model_id in MODEL_IDS for r in clean[model_id].values()])
    assert len(frame)==3642 and not frame.duplicated(['model','sample_id']).any()
    atomic_csv(ROOT/'clean_test_predictions.csv',frame)
    atomic_json(ROOT/'clean_test_metrics.json',dict(experiment_fingerprint=EXPERIMENT_FP,split='TEST',models=metrics))
    atomic_json(ROOT/'inference_diagnostics.json',dict(experiment_fingerprint=EXPERIMENT_FP,models=diagnostics))
    return clean,metrics


def runtime_preflight():
    """Kaggle-only, no optimization/search: validate restored paths and pinned API contracts."""
    assert Path('/kaggle/working').is_dir() and PREFLIGHT_OK
    assert all(installed_version(k)==v for k,v in PINS.items())
    assert torch.cuda.is_available() and DEVICE.type=='cuda'
    assert len(TRAIN_DS)==6920 and len(VALIDATION_DS)==872 and len(TEST_ROWS)==1821
    for split,size in EXPECTED_SIZES.items():
        rows=SPLIT_ROWS[split]
        assert len(rows)==size and {r['label'] for r in rows}=={0,1}
        assert stable_hash(rows)==DATA_MANIFEST['splits'][split]['content_sha256']
    assert 'num_items_in_batch' in inspect.signature(Trainer.compute_loss).parameters
    assert {'eval_strategy','save_strategy','gradient_accumulation_steps','processing_class'} <= (
        set(inspect.signature(TrainingArguments).parameters)|set(inspect.signature(Trainer).parameters))
    assert all(hasattr(TrainerCallback,name) for name in ('on_epoch_end','on_log','on_save'))
    checked_paths=[]
    # Validate path/schema relationships now; full weight hashes are still verified immediately before reuse.
    for model_id in MODEL_IDS:
        best=winner_directory(model_id)
        if RUN_STAGE in {'evaluate','attack'} or best.exists():
            for name in ('artifact_manifest.json','config.json','model.safetensors','tokenizer_config.json','provenance.json'):
                assert (best/name).is_file(), f'Required winning checkpoint path missing: {best/name}'
            manifest=unseal(read_json(best/'artifact_manifest.json'))
            assert manifest['experiment_fingerprint']==EXPERIMENT_FP
            assert read_json(best/'provenance.json')['source_sha256']==SOURCE_HASHES
            checked_paths.append(str(best))
        for lr,batch in GRID:
            assert batch%min(MAX_MICROBATCH,batch)==0
            candidate=ROOT/'candidates'/model_id/f'lr_{lr:.0e}_bs_{batch}'
            for checkpoint in (candidate/'checkpoints').glob('checkpoint-*'):
                commit=checkpoint/'checkpoint_commit.json'
                if not commit.exists():
                    LOG.warning('Preflight found uncommitted interrupted save: %s (not resumable)',checkpoint)
                    continue
                saved=unseal(read_json(commit)); state=read_json(checkpoint/'trainer_state.json')
                assert state['global_step']==saved['step']
                assert all((checkpoint/k).is_file() for k in saved['files']), f'Incomplete committed path: {checkpoint}'
                selected=state.get('best_model_checkpoint')
                if selected:
                    selected=Path(selected)
                    assert selected.resolve().parent==(candidate/'checkpoints').resolve(), 'Selected checkpoint points outside its candidate'
                    assert (selected/'model.safetensors').is_file(), f'Selected checkpoint is unavailable: {selected}'
                checked_paths.append(str(checkpoint))
    if RUN_STAGE=='attack':
        for model_id in MODEL_IDS:
            assert (ROOT/'clean_records'/model_id/'complete.json').is_file(), 'Run evaluate before attack'
    with tempfile.TemporaryDirectory(prefix='path_preflight_',dir=ROOT) as directory:
        probe=Path(directory)/'atomic_probe.json'
        atomic_json(probe,seal({'experiment_fingerprint':EXPERIMENT_FP}))
        assert unseal(read_json(probe))['experiment_fingerprint']==EXPERIMENT_FP
    record=dict(experiment_fingerprint=EXPERIMENT_FP,stage=RUN_STAGE,status='passed',
        dependency_versions=PINS,gpu=torch.cuda.get_device_name(0),split_sizes=EXPECTED_SIZES,
        checkpoint_paths=checked_paths,recipe_and_model_preflight_passed=PREFLIGHT_OK,
        trainer='4.57.6 API/callback contracts checked; actual optimizer steps and sample windows recorded at runtime',
        optimization_steps_in_preflight=0,diagnostics='Prior preflight verified forward/save-reload and TextAttack cached logical-versus-physical queries')
    atomic_json(ROOT/'runtime_preflight.json',record)
    LOG.info('Kaggle runtime preflight passed: dependencies, dataset integrity, model/recipe probes, APIs and %d restored checkpoint paths',len(checked_paths))
    if RUN_STAGE=='all' and HYPERPARAM_MODE=='paper_grid':
        LOG.warning('RUN_STAGE=all schedules 18 candidates and four full attacks: high single-session risk. Prefer saved train/evaluate/attack stages; paper_grid remains unchanged.')


## 11. Full attacks and interruption-safe per-example journal

Four jobs share the same original test IDs: baseline/TextFooler, baseline/TextBugger, PuRe/TextFooler, PuRe/TextBugger. Each builds a fresh victim-specific attack and generates its own adversarial texts. No adversarial example is transferred between victims. Every clean-correct example is scheduled; clean-incorrect examples are accounted for as ineligible.

An immutable ordered event journal commits one final attempt result **and its post-attempt RNG state atomically**. On restart, rebuild per-example state from verified events, restore the last committed stream, clear caches, and continue. A separate in-flight marker identifies a hard-killed attempt; its query cost is unknown and explicitly recorded when retried. Errors retry up to the configured bound, with each attempt preserved. Errors never become failed attacks. Stochastic attack-initial mismatches are unresolved skips and are **not automatically redrawn** to obtain a favorable clean prediction; they require interpretation through the separate diagnostics. They do not count as valid completed coverage. Each mismatch logs the sample ID, frozen clean prediction, attack-initial prediction, gold label, probabilities and query count without an extra prediction call. Even one such unresolved eligible example keeps full-test ASR/AUA null; conditional metrics and lower/upper bounds remain descriptive only. Restarting does not redraw a mismatch.

`changed_words` uses TextAttack's `words_diff_num`; percentage changed is that word edit count / original TextAttack word count × 100 (null for zero words). Every 100 committed per-example attempt results export all four full-population CSV snapshots, conditional metrics and an INCOMPLETE report; every example is already persisted before that snapshot. KeyboardInterrupt exports in `finally`; a hard kill can leave CSV snapshots behind the journal. On restart run the final export cell to reconstruct the latest committed partial outputs. No finite per-example time limit or total-example limit is imposed.


In [11]:
# These globals are populated only after BOTH clean test evaluations have completed.
CLEAN, CLEAN_METRICS, RUN_FINGERPRINT = None, None, None

def bind_attack_run():
    global CLEAN,CLEAN_METRICS,RUN_FINGERPRINT
    CLEAN,CLEAN_METRICS=load_complete_clean()
    winners={m:verify_model_directory(winner_directory(m)) for m in MODEL_IDS}
    clean_checks={m:unseal(read_json(ROOT/'clean_records'/m/'complete.json')) for m in MODEL_IDS}
    RUN_FINGERPRINT=stable_hash(dict(experiment=EXPERIMENT_FP,winners=winners,clean=clean_checks,
                                   recipes=RECIPE_MANIFEST,seed=SEED))
    # Rebuild top-level clean outputs from authoritative records if a prior interruption
    # occurred after clean completion but before consolidated CSV/JSON export.
    frame=pd.DataFrame([{k:v for k,v in r.items() if k not in {'rng_after','clean_fingerprint'}}
                        for m in MODEL_IDS for r in CLEAN[m].values()])
    atomic_csv(ROOT/'clean_test_predictions.csv',frame)
    atomic_json(ROOT/'clean_test_metrics.json',dict(experiment_fingerprint=EXPERIMENT_FP,split='TEST',models=CLEAN_METRICS))
    atomic_json(ROOT/'attack_run_manifest.json',dict(run_fingerprint=RUN_FINGERPRINT,
        experiment_fingerprint=EXPERIMENT_FP,seed=SEED,jobs=ALL_JOBS,
        test_sample_ids=[r['sample_id'] for r in TEST_ROWS],N_total=1821,
        common_correct_sample_ids=[s['sample_id'] for s in TEST_ROWS if all(CLEAN[m][s['sample_id']]['correct'] for m in MODEL_IDS)]))

def job_directory(model_id,attack_name):
    return ROOT/'attack_records'/f'{model_id}_{attack_name}'

def base_attack_row(model_id,attack_name,sample):
    clean=CLEAN[model_id][sample['sample_id']]
    return dict(run_fingerprint=RUN_FINGERPRINT,experiment_fingerprint=EXPERIMENT_FP,seed=SEED,
        model=model_id,attack=attack_name,sample_id=sample['sample_id'],original_text=sample['text'],
        original_text_sha256=sample['original_text_sha256'],label=sample['label'],
        clean_prediction=clean['prediction'],initially_correct=clean['correct'],
        attack_status='pending' if clean['correct'] else 'ineligible',attempted=False,
        attack_initial_prediction=None,adversarial_text=None,adversarial_prediction=None,
        changed_words=None,original_word_count=None,changed_word_pct=None,num_queries=None,
        native_result_num_queries=None,physical_queries=None,elapsed_seconds=0.0,
        attack_initial_probabilities=None,attack_initial_probability_for_label=None,initial_probability_delta=None,
        attempts=0,retries=0,error=None,total_known_queries_all_attempts=0,
        total_known_runtime_all_attempts=0.0,interrupted_attempt_cost_unknown=False)

def read_job(model_id,attack_name):
    directory=job_directory(model_id,attack_name)
    rows={s['sample_id']:base_attack_row(model_id,attack_name,s) for s in TEST_ROWS}
    events=[]
    for i,p in enumerate(sorted(directory.glob('event_*.json')),start=1):
        assert p.name==f'event_{i:06d}.json', 'Attack journal gap/duplicate sequence'
        event=unseal(read_json(p))
        assert event['sequence']==i and event['run_fingerprint']==RUN_FINGERPRINT
        assert event['model']==model_id and event['attack']==attack_name
        r=event['result']; sid=r['sample_id']; assert sid in rows
        previous=rows[sid]
        assert previous['initially_correct'] and previous['attack_status'] not in {'success','failed','initial_mismatch'}, 'Duplicate resolved attack or forbidden mismatch redraw'
        assert r['attempts']==previous['attempts']+1
        assert r['model']==model_id and r['attack']==attack_name and r['run_fingerprint']==RUN_FINGERPRINT
        assert r['original_text']==SAMPLE_BY_ID[sid]['text'] and r['label']==SAMPLE_BY_ID[sid]['label']
        assert r['attack_status'] in {'success','failed','error','initial_mismatch'} and r['attempted']
        if r['attack_status'] in {'success','failed'}:
            assert r['attack_initial_prediction']==r['label'], 'Eligible attack began incorrectly'
            assert isinstance(r['num_queries'],int) and r['num_queries']>=1
            assert isinstance(r['adversarial_text'],str)
            assert (r['adversarial_prediction']!=r['label']) == (r['attack_status']=='success')
        rows[sid]=r; events.append(event)
    deduplicate_records(list(rows.values()))
    assert len(rows)==1821
    return rows,events

def run_attack_job(model_id,attack_name):
    assert RUN_FINGERPRINT and CLEAN is not None, 'Both complete clean test runs are required first'
    directory=job_directory(model_id,attack_name); directory.mkdir(parents=True,exist_ok=True)
    rows,events=read_job(model_id,attack_name)
    wrapper=load_victim(model_id); attack=build_attack(attack_name,wrapper)
    inflight=directory/'inflight.json'
    try:
        if events: restore_rng(events[-1]['rng_after'])
        else: set_seed(SEED+5000+ALL_JOBS.index((model_id,attack_name))) # seed whole job, not each text
        interrupted=None
        if inflight.exists():
            interrupted=unseal(read_json(inflight))
            assert interrupted['run_fingerprint']==RUN_FINGERPRINT
            if interrupted['sequence']<=len(events):
                inflight.unlink(); interrupted=None # event committed just before interruption
            else:
                assert interrupted['sequence']==len(events)+1
                restore_rng(interrupted['rng_before'])
                LOG.warning('Retrying interrupted in-flight sample %s; its previous cost is unknown',interrupted['sample_id'])
        # Passes retry infrastructure errors only; mismatches remain explicitly unresolved.
        for retry_pass in range(MAX_ATTEMPTS):
            schedule=TEST_ROWS
            if interrupted is not None:
                schedule=[SAMPLE_BY_ID[interrupted['sample_id']]]+[s for s in TEST_ROWS if s['sample_id']!=interrupted['sample_id']]
            for sample in tqdm(schedule,desc=f'{model_id}/{attack_name} pass {retry_pass+1}'):
                sid=sample['sample_id']; previous=rows[sid]
                status=previous['attack_status']
                if status in {'ineligible','success','failed','initial_mismatch'}: continue
                if previous['attempts']>=MAX_ATTEMPTS: continue
                assert status in {'pending','error'} and previous['initially_correct']
                if interrupted is not None:
                    # Match the interrupted sample before consuming any other RNG draws.
                    assert sid==interrupted['sample_id'], 'Interrupted job scheduling differs; inspect lineage'
                attack.clear_cache(); wrapper.physical_queries=0
                before=capture_rng(); sequence=len(events)+1
                atomic_json(inflight,seal(dict(run_fingerprint=RUN_FINGERPRINT,model=model_id,attack=attack_name,
                    sample_id=sid,sequence=sequence,rng_before=before)))
                t=time.perf_counter(); row=dict(previous)
                row.update(attempted=True,attempts=previous['attempts']+1,retries=previous['attempts'],
                    attack_initial_prediction=None,native_result_num_queries=None,
                    attack_initial_probabilities=None,attack_initial_probability_for_label=None,initial_probability_delta=None,
                    error=None,interrupted_attempt_cost_unknown=previous['interrupted_attempt_cost_unknown'] or interrupted is not None)
                attack.goal_function.num_queries=0
                try:
                    result=attack.attack(sample['text'],int(sample['label']))
                    original=result.original_result; perturbed=result.perturbed_result
                    initial_pred=int(original.output)
                    row['attack_initial_prediction']=initial_pred
                    initial_probabilities=np.asarray(original.raw_output).astype(float)
                    row['attack_initial_probabilities']=initial_probabilities.tolist()
                    row['attack_initial_probability_for_label']=float(initial_probabilities[sample['label']])
                    clean=CLEAN[model_id][sid]
                    clean_probability=clean['positive_probability'] if sample['label'] else clean['negative_probability']
                    row['initial_probability_delta']=row['attack_initial_probability_for_label']-clean_probability
                    row['native_result_num_queries']=int(result.num_queries)
                    row['num_queries']=int(attack.goal_function.num_queries) # final search counter, not earlier best candidate's count
                    assert 1<=row['native_result_num_queries']<=row['num_queries'], 'Inconsistent TextAttack query counters'
                    row['adversarial_text']=perturbed.attacked_text.text
                    row['adversarial_prediction']=int(perturbed.output)
                    words=int(original.attacked_text.num_words)
                    changed=int(original.attacked_text.words_diff_num(perturbed.attacked_text))
                    row.update(changed_words=changed,original_word_count=words,changed_word_pct=pct(changed,words))
                    if isinstance(result,SkippedAttackResult) or initial_pred!=sample['label']:
                        row.update(attack_status='initial_mismatch',error='Attack-time original prediction disagrees with frozen clean eligibility; unresolved, not failure')
                        LOG.warning('%s/%s initial mismatch sample=%s clean=%d attack_initial=%d gold=%d probabilities=%s delta_gold_probability=%.6g queries=%d; no redraw; full-test ASR/AUA remain incomplete',
                            model_id,attack_name,sid,clean['prediction'],initial_pred,sample['label'],
                            row['attack_initial_probabilities'],row['initial_probability_delta'],row['num_queries'])
                    elif isinstance(result,SuccessfulAttackResult): row['attack_status']='success'
                    elif isinstance(result,FailedAttackResult): row['attack_status']='failed'
                    else: raise RuntimeError(f'Unexpected result type: {type(result).__name__}')
                    if row['attack_status'] in {'success','failed'}:
                        assert (row['adversarial_prediction']!=sample['label'])==(row['attack_status']=='success')
                except (KeyboardInterrupt,SystemExit):
                    # Do not commit an unknown result. In-flight marker + prior RNG permits exact restart.
                    raise
                except Exception as exc:
                    row.update(attack_status='error',error=f'{type(exc).__name__}: {exc}',
                        num_queries=int(attack.goal_function.num_queries),adversarial_text=None,
                        adversarial_prediction=None,changed_words=None,original_word_count=None,changed_word_pct=None)
                    LOG.exception('Attack error for %s',sid)
                finally:
                    row['elapsed_seconds']=time.perf_counter()-t
                    row['physical_queries']=int(wrapper.physical_queries)
                row['total_known_queries_all_attempts']=previous['total_known_queries_all_attempts']+(row['num_queries'] or 0)
                row['total_known_runtime_all_attempts']=previous['total_known_runtime_all_attempts']+row['elapsed_seconds']
                event=dict(sequence=sequence,run_fingerprint=RUN_FINGERPRINT,model=model_id,attack=attack_name,
                    result=row,rng_after=capture_rng())
                atomic_json(directory/f'event_{sequence:06d}.json',seal(event))
                rows[sid]=row; events.append(event); interrupted=None
                inflight.unlink(missing_ok=True)
                if sequence%FLUSH_EVERY==0:
                    save_attack_outputs(make_plots=False)
                if row['attack_status']=='error': free_gpu()
            if not any(r['attack_status']=='error' and r['attempts']<MAX_ATTEMPTS for r in rows.values()): break
        unresolved=[r for r in rows.values() if r['initially_correct'] and r['attack_status'] not in {'success','failed'}]
        if unresolved:
            LOG.warning('%s/%s INCOMPLETE: %d unresolved eligible examples. Initial mismatches remain unresolved; infrastructure errors retry only up to MAX_ATTEMPTS=%d.',model_id,attack_name,len(unresolved),MAX_ATTEMPTS)
    finally:
        del attack,wrapper; free_gpu()
        save_attack_outputs(make_plots=False)


## 12. Full-population metrics and common-correct paired comparison

For each job, `Clean Acc = correct/1821`, `ASR = successes/correct`, `AUA = (correct-successes)/1821`. Definitive ASR and AUA are null until eligible coverage is complete. Check `AUA = Clean Acc × (1−ASR)` in fraction units. With unresolved examples, report conditional ASR on resolved outcomes, ASR bounds `[successes,successes+unresolved]/correct`, and AUA bounds `[failed,correct-successes]/1821`.

The paired set contains exactly those test IDs both winners initially classify correctly. Its ASR denominator is that common-correct count; survival uses the same denominator, not 1,821. Paired definitive values require resolution for both victims on that set. Numerators/denominators and bounds are exported. Attack runtimes sum measured attempt time and omit offline downtime; hard-killed in-flight cost can be unknown. Successful adversarial texts are examples under automated constraints, not evidence that human sentiment labels are preserved.


In [12]:
def finite_mean(values):
    return float(np.mean(values)) if values else None

def job_metrics(model_id,attack_name,rows):
    values=list(rows.values()); correct=sum(r['initially_correct'] for r in values)
    m=count_metrics(1821,correct,[r['attack_status'] for r in values],[r['attempted'] for r in values])
    attempted=[r for r in values if r['attempted']]
    resolved=[r for r in values if r['attack_status'] in {'success','failed'}]
    successful=[r for r in values if r['attack_status']=='success']
    queries=[r['num_queries'] for r in attempted if r['num_queries'] is not None]
    changes=[r['changed_word_pct'] for r in resolved if r['changed_word_pct'] is not None]
    m.update(model=model_id,attack=attack_name,seed=SEED,split='TEST',run_fingerprint=RUN_FINGERPRINT,
        status='COMPLETE' if m['complete'] else 'INCOMPLETE',
        initial_prediction_disagreement_count=sum(r['attack_status']=='initial_mismatch' for r in values),
        initial_prediction_disagreement_denominator=sum(r['attack_initial_prediction'] is not None for r in values),
        N_retry_exhausted=sum(r['attack_status']=='error' and r['attempts']>=MAX_ATTEMPTS for r in values),
        initial_prediction_disagreement_pct=pct(sum(r['attack_status']=='initial_mismatch' for r in values),
            sum(r['attack_initial_prediction'] is not None for r in values)),
        avg_queries_all_attacked=finite_mean(queries) if len(queries)==len(attempted) else None,
        avg_queries_denominator=len(attempted),queries_sum=sum(queries),queries_known_count=len(queries),
        avg_queries_successful=finite_mean([r['num_queries'] for r in successful]),
        avg_queries_successful_denominator=len(successful),
        avg_queries_resolved=finite_mean([r['num_queries'] for r in resolved]),
        median_queries_all_attacked=float(np.median(queries)) if queries and len(queries)==len(attempted) else None,
        avg_changed_word_pct=finite_mean(changes),changed_word_denominator=len(changes),
        avg_changed_word_pct_successful=finite_mean([r['changed_word_pct'] for r in successful if r['changed_word_pct'] is not None]),
        attack_runtime_seconds=sum(r['total_known_runtime_all_attempts'] for r in values),
        final_attempt_runtime_seconds=sum(r['elapsed_seconds'] for r in values),
        total_known_queries_including_retries=sum(r['total_known_queries_all_attempts'] for r in values),
        total_attempts=sum(r['attempts'] for r in values),total_retries=sum(r['retries'] for r in values),
        interrupted_cost_unknown=any(r['interrupted_attempt_cost_unknown'] for r in values))
    return m

def paired_metrics(frames):
    common=[s['sample_id'] for s in TEST_ROWS if all(CLEAN[m][s['sample_id']]['correct'] for m in MODEL_IDS)]
    results=[]
    for name in ATTACK_NAMES:
        both_complete=all(frames[(m,name)][sid]['attack_status'] in {'success','failed'} for m in MODEL_IDS for sid in common)
        for m in MODEL_IDS:
            selected=[frames[(m,name)][sid] for sid in common]
            s=sum(r['attack_status']=='success' for r in selected)
            f=sum(r['attack_status']=='failed' for r in selected); n=len(selected); u=n-s-f
            results.append(dict(model=m,attack=name,denominator_name='common clean-correct test IDs in both models',
                denominator=n,successful=s,failed=f,unresolved=u,paired_complete=both_complete,
                asr_pct=pct(s,n) if both_complete else None,survival_pct=pct(n-s,n) if both_complete else None,
                asr_numerator=s,survival_numerator=n-s if both_complete else None,
                asr_lower_pct=pct(s,n),asr_upper_pct=pct(s+u,n),
                survival_lower_pct=pct(f,n),survival_upper_pct=pct(n-s,n)))
    return results

# Reference-only values: used nowhere in model selection, training, prediction or attack scheduling.
PAPER_REFERENCE={
 ('bert_baseline','textfooler'):dict(clean_accuracy_pct=92.09,aua_pct=6.32,asr_pct=93.14,avg_queries_all_attacked=87.07),
 ('bert_baseline','textbugger'):dict(clean_accuracy_pct=92.09,aua_pct=30.37,asr_pct=67.02,avg_queries_all_attacked=41.19),
 ('bert_pure','textfooler'):dict(clean_accuracy_pct=90.88,aua_pct=30.37,asr_pct=66.59,avg_queries_all_attacked=134.01),
 ('bert_pure','textbugger'):dict(clean_accuracy_pct=90.88,aua_pct=47.17,asr_pct=48.10,avg_queries_all_attacked=58.52)}

def paper_comparison(summaries):
    comparison=[]
    for row in summaries:
        reference=PAPER_REFERENCE[(row['model'],row['attack'])]
        for metric,paper in reference.items():
            actual=row[metric]
            # A partial query average is descriptive only and does not become a full-test comparison.
            comparable=(metric=='clean_accuracy_pct' or row['complete'])
            measured=actual if comparable else None
            comparison.append(dict(model=row['model'],attack=row['attack'],metric=metric,
                actual=measured,paper_reference=paper,
                actual_minus_paper=actual_minus_paper(measured,paper),
                difference_unit='queries' if metric.startswith('avg_queries') else 'percentage points',
                eligible_coverage_pct=row['eligible_coverage_pct'],complete=row['complete'],
                note='reference only; never an experimental output substitution'))
    for row in comparison:
        if row['actual'] is not None:
            assert math.isclose(row['actual_minus_paper'],row['actual']-row['paper_reference'],abs_tol=1e-10)
    return comparison


## 13. Plots, comparison tables and research report

All measured series are computed from records. Missing complete metrics are labeled INCOMPLETE and are not replaced with reference values. Figures cover clean/AUA, ASR, query averages, training/validation curves, success/failure/unresolved counts and clean confusion matrices. The paper comparison uses percentage-point differences for accuracies/rates and query-unit differences for AvgQ.

The report identifies data provenance uncertainty, unspecified paper settings, batching/padding and stochasticity, library/default versions, query definitions and optional deviations. Neither success under these recipes nor higher AvgQ demonstrates robustness to adaptive expectation-over-transformation attacks; this notebook reproduces the specified recipe evaluation and reports the randomized-defense limitation explicitly.


In [13]:
def plot_results(summaries,frames):
    plot_dir=ROOT/'plots'; plot_dir.mkdir(exist_ok=True)
    def finish(fig,name):
        fig.tight_layout(); temp=plot_dir/(name+'.tmp')
        fig.savefig(temp,format='png',dpi=170,bbox_inches='tight'); plt.close(fig)
        os.replace(temp,plot_dir/name)
    labels=[f'{s["model"]}\n{s["attack"]}' for s in summaries]; x=np.arange(4)
    fig,ax=plt.subplots(figsize=(11,4))
    ax.bar(x-.18,[s['clean_accuracy_pct'] for s in summaries],.36,label='Clean TEST (all 1821)')
    for i,s in enumerate(summaries):
        if s['aua_pct'] is None:
            ax.plot([i+.18,i+.18],[s['aua_lower_pct'],s['aua_upper_pct']],color='darkorange',lw=3)
            ax.text(i+.18,s['aua_upper_pct']+2,'INCOMPLETE',ha='center',fontsize=7,rotation=90)
        else: ax.bar(i+.18,s['aua_pct'],.36,color='darkorange',label='AUA' if i==0 else None)
    ax.set_xticks(x,labels); ax.set_ylim(0,110); ax.set_ylabel('Accuracy / AUA (%)'); ax.legend()
    ax.set_title('Measured clean accuracy and full-test AUA (intervals if incomplete)')
    finish(fig,'clean_vs_aua.png')
    for key,ylabel,filename in [('asr_pct','ASR (%)','asr.png'),
                               ('avg_queries_all_attacked','Mean logical queries / attempted example','avg_queries.png')]:
        fig,ax=plt.subplots(figsize=(11,4))
        for i,s in enumerate(summaries):
            value=s[key]
            if value is not None:
                ax.bar(i,value,color='steelblue' if s['complete'] else 'lightgray')
                if not s['complete']: ax.text(i,value,'conditional / incomplete',ha='center',va='bottom',fontsize=7)
            elif key=='asr_pct' and s['asr_lower_pct'] is not None:
                ax.plot([i,i],[s['asr_lower_pct'],s['asr_upper_pct']],lw=3,color='darkorange')
                ax.text(i,s['asr_upper_pct'],'INCOMPLETE',ha='center',va='bottom',fontsize=7)
            else: ax.text(i,0,'No measured attempts',rotation=90,va='bottom',ha='center')
        ax.set_xticks(x,labels); ax.set_ylabel(ylabel); ax.set_title('Measured '+ylabel)
        finish(fig,filename)
    fig,ax=plt.subplots(figsize=(11,4)); bottom=np.zeros(4)
    for key,label,color in [('N_successful','successful','firebrick'),('N_failed','failed','seagreen'),
                             ('N_unresolved','unresolved','darkorange'),('N_initially_incorrect','ineligible','gray')]:
        values=np.array([s[key] for s in summaries]); ax.bar(x,values,bottom=bottom,label=label,color=color); bottom+=values
    assert all(bottom==1821)
    ax.set_xticks(x,labels); ax.set_ylabel('Test sample count'); ax.set_title('Full-test attack outcomes: every job totals 1821'); ax.legend()
    finish(fig,'attack_success_counts.png')
    history_path=ROOT/'training_history.csv'
    if history_path.exists() and history_path.stat().st_size > 0:
        h=pd.read_csv(history_path)
        if not {'model', 'candidate_id'}.issubset(h.columns):
            LOG.warning('Training history has no candidate rows yet; skipping training_curves.png')
            h=pd.DataFrame(columns=['model','candidate_id'])
        fig,axes=plt.subplots(1,3,figsize=(15,4))
        for (m,cid),group in h.groupby(['model','candidate_id']):
            if 'eval_accuracy' in group:
                e=group.dropna(subset=['eval_accuracy']); axes[0].plot(e['epoch'],100*e['eval_accuracy'],alpha=.6,label=f'{m}/{cid}')
            if 'eval_loss' in group:
                e=group.dropna(subset=['eval_loss']); axes[1].plot(e['epoch'],e['eval_loss'],alpha=.6)
            key='train_epoch_loss' if 'train_epoch_loss' in group else 'loss'
            if key in group:
                e=group.dropna(subset=[key]); axes[2].plot(e['epoch'],e[key],alpha=.6)
        axes[0].set_title('Validation accuracy (%)'); axes[1].set_title('Validation cross-entropy'); axes[2].set_title('Training loss')
        for ax in axes: ax.set_xlabel('Epoch')
        axes[0].legend(fontsize=5,ncol=2); finish(fig,'training_curves.png')
    fig,axes=plt.subplots(1,2,figsize=(9,4))
    for ax,m in zip(axes,MODEL_IDS):
        cm=np.array(CLEAN_METRICS[m]['confusion_matrix']); ax.imshow(cm,cmap='Blues')
        for (i,j),v in np.ndenumerate(cm): ax.text(j,i,str(v),ha='center',va='center')
        ax.set_xticks([0,1],['negative','positive']); ax.set_yticks([0,1],['negative','positive'])
        ax.set_xlabel('Predicted'); ax.set_ylabel('Gold'); ax.set_title(m+' TEST, N=1821')
    finish(fig,'clean_confusion_matrices.png')

def markdown_table(rows,columns):
    def text(v):
        if v is None: return '—'
        if isinstance(v,float): return f'{v:.4f}'
        return str(v).replace('|',chr(92)+'|').replace('\n',' ')
    lines=['| '+' | '.join(columns)+' |','| '+' | '.join(['---']*len(columns))+' |']
    lines+=['| '+' | '.join(text(r.get(c)) for c in columns)+' |' for r in rows]
    return '\n'.join(lines)

def protocol_deviations():
    notes=[
        'Candidate dataset: SetFit/sst2 immutable revision; matching Table 8 counts does not prove identical authors raw-data provenance.',
        'Implementation-chosen seed 42 (or configured seed), max_length=128, dropout from pretrained config, weight_decay=0.01, AdamW betas=(0.9,0.999), eps=1e-8, linear schedule and no warmup; these are not asserted to be paper-specified.',
        'Physical training microbatch<=8, effective batch via accumulation, gradient checkpointing and FP32 chosen for T4 memory and randomized PCA support; no silent OOM adjustments.',
        'Singleton validation, clean and attack victim forwards avoid batch-dependent padding lengths; official pooling is unchanged. Paper does not fully specify padding/inference batching.',
        'Pinned current-compatible library environment is an implementation choice; paper does not pin TextAttack/default versions, seed, optimizer details, scheduler, tokenization limit or best-epoch selection rule.',
        'TextAttack default within-example model cache retained; between-example caches cleared. Logical recipe query counts can differ from physical forwards. AvgQ denominator in paper is not fully disambiguated, so all-attempt and successful-only variants are both exported.',
        'PuRe randomized SVD remains stochastic. Frozen clean eligibility and later initial queries may disagree; those examples are unresolved, not successful/failed attacks. No automatic mismatch redraws.',
        'Automated semantic constraints do not establish preservation of human sentiment labels. No transfer attacks are used; no adaptive EOT robustness claim is made.',
        'Single-seed results cannot establish variance or statistical equivalence to the published row; separate seed lineages are supported.',
        'Post-interruption checkpoint/journal RNG resume is verified, but hard-killed uncommitted attempt costs and wall time may be unknown. Bitwise equivalence across hardware/kernel versions is not promised.'
    ]
    if HYPERPARAM_MODE=='fixed': notes.append('PROTOCOL APPROXIMATION: fixed lr=2e-5, effective batch=16 replaces the full nine-configuration grid.')
    if QUERY_CAP is not None: notes.append(f'ATTACK DEVIATION: explicit query cap {QUERY_CAP} replaces the unlimited recipe default.')
    if RNG_POLICY!='paper_like_stochastic': notes.append('SENSITIVITY ONLY: deterministic_diagnostic requests strict deterministic kernels and whole-stream replay; randomized SVD remains state-dependent. Neither policy reseeds individual texts.')
    return notes

def write_reproduction_report(summaries=None,comparison=None,paired=None):
    training=read_json(ROOT/'training_summary.json') if (ROOT/'training_summary.json').exists() else {'status':'NOT_RUN','selections':{}}
    complete=summaries is not None and all(s['complete'] for s in summaries)
    lines=['# PuRe BERT-base SST-2 reproduction report','',
        '**'+('COMPLETE ELIGIBLE COVERAGE — measured recipe evaluation' if complete else 'INCOMPLETE — do not report partial ASR as a full-test result')+'**','',
        f'Experiment fingerprint: `{EXPERIMENT_FP}`',f'Seed: {SEED}; stage: {RUN_STAGE}; RNG: {RNG_POLICY}; hyperparameters: {HYPERPARAM_MODE}.','',
        'Paper: https://aclanthology.org/2025.tacl-1.63.pdf',
        f'Official source: https://github.com/PuReDefence/PuRe/tree/{PURE_COMMIT}',
        f'Dataset: https://huggingface.co/datasets/SetFit/sst2/tree/{DATASET_COMMIT}',
        '','## Protocol and chosen settings','',
        'Train=6920, validation=872, labeled TEST=1821. Only train is optimized; only validation selects candidates/checkpoints.',
        'All candidates train all four epochs; ties use validation accuracy, earliest epoch, lower learning rate, smaller effective batch.',
        'Baseline and PuRe start independently from the same untouched pretrained embeddings/encoder. PuRe constructor settings are passed explicitly on reload.',
        '```json',json.dumps(dict(training=TRAIN_SETTINGS,pure=PURE_KWARGS,implementation_defaults=IMPLEMENTATION_DEFAULTS,
            learning_rates=sorted({lr for lr,b in GRID}),batch_sizes=sorted({b for lr,b in GRID})),indent=2),'```','',
        '## Validation selection (not TEST)','',f'Training status: {training.get("status")}.','']
    audit_summary=DATA_MANIFEST.get('overlap_summary',{})
    audit_lines=['','## Canonical dataset overlap and label audit','',
        markdown_table([audit_summary],['duplicate_groups','within_split_duplicate_groups',
            'across_split_groups','train_test_groups','contradictory_groups','duplicate_occurrences']),
        'Cross-split normalized-text group counts: '+json.dumps(audit_summary.get('cross_split_group_counts',{}),sort_keys=True),
        'Potential evaluation leakage: '+str(DATA_MANIFEST.get('potential_evaluation_leakage',False))+'. '
        'Potential label inconsistency: '+str(DATA_MANIFEST.get('potential_label_inconsistency',False))+'.',
        'Original canonical SetFit/sst2 splits, row order, texts and labels are preserved. No examples are removed, moved, deduplicated or relabeled. '
        'Cross-split overlap may affect validation selection and test estimates; contradictory labels are not treated as harmless. '
        'Inspect dataset_overlap_report.json for every group, sample ID, split, provided label and original text.']
    audit_lines+=['WARNING: '+message for message in DATA_MANIFEST.get('dataset_warnings',audit_summary.get('warnings',[]))]
    audit_index=lines.index('## Validation selection (not TEST)')
    lines[audit_index:audit_index]=audit_lines
    selections=[dict(model=m,**{k:v for k,v in r.items() if k!='model'}) for m,r in training.get('selections',{}).items()]
    if selections: lines.append(markdown_table(selections,['model','lr','batch_size','selected_epoch','validation_accuracy','validation_loss','optimizer_steps','completed_session_wall_seconds']))
    if summaries is not None:
        lines+=['','## Clean TEST metrics','',markdown_table([dict(model=m,**v) for m,v in CLEAN_METRICS.items()],
            ['model','N','N_correct','accuracy_pct','cross_entropy_loss','precision_pct','recall_pct','f1_pct','macro_f1_pct']),
            '','## Independent attacks on the complete test population','',
            markdown_table(summaries,['model','attack','status','N_total','N_initially_correct','N_initially_incorrect',
                'N_attacked','N_successful','N_failed','N_skipped','N_error','N_pending','eligible_coverage_pct']),
            '',markdown_table(summaries,['model','attack','clean_accuracy_pct','asr_pct','aua_pct','avg_queries_all_attacked',
                'avg_queries_successful','median_queries_all_attacked','avg_changed_word_pct','attack_runtime_seconds']),
            '',markdown_table(summaries,['model','attack','conditional_resolved_asr_pct','conditional_resolved_denominator',
                'asr_lower_pct','asr_upper_pct','aua_lower_pct','aua_upper_pct']),
            '','Definitive ASR numerator=successful, denominator=initially correct. AUA denominator=1821. Percentages are not fractions.',
            'AvgQ includes every eligible attempted final attempt, including error/skip counters; successful-only and resolved-only variants are separate. Retry cost is exported separately.',
            '', '### Attack-initial stochastic mismatches', '',
            markdown_table(summaries,['model','attack','initial_prediction_disagreement_count',
                'initial_prediction_disagreement_denominator','initial_prediction_disagreement_pct','eligible_coverage_pct']),
            'A mismatch compares the single attack-initial draw with frozen clean eligibility. It is an unresolved skip, never a success/failure, and is not redrawn on restart. Any unresolved eligible mismatch prevents definitive full-test ASR/AUA; inspect bounds and sensitivity diagnostics.',
            '','## Paired common-correct set','',markdown_table(paired,['model','attack','denominator','successful','unresolved','paired_complete','asr_pct','survival_pct']),
            '','## Table 1 reference comparison','',markdown_table(comparison,['model','attack','metric','actual','paper_reference','actual_minus_paper','difference_unit']),
            '','Reference values are comparisons only; absent measurements remain null. Complete coverage does not itself demonstrate numerical reproduction of the published values.']
        diagnostics_path=ROOT/'inference_diagnostics.json'
        if diagnostics_path.exists():
            diagnostics=read_json(diagnostics_path)['models']
            lines+=['','## Randomization and padding sensitivity','',markdown_table(list(diagnostics.values()),
                ['model','N','repeats','repeated_disagreement_pct','singleton_vs_dynamic_batch_disagreement_pct',
                 'singleton_vs_fixed_padding_same_rng_disagreement_pct','fixed_padding_mean_max_probability_delta'])]
    else:
        lines+=['','TEST/attacks have not completed in this stage. No measured test result or attack metric is implied.']
    lines+=['','## Deviations, ambiguities and possible discrepancy sources','']
    lines+=['- '+s for s in protocol_deviations()]
    lines+=['','## Resume and artifacts','',
        'The authoritative per-example ledgers and winning/candidate models are outside the compact report ZIP. Save the complete output tree as a Kaggle version/dataset and reattach it for cross-session resume. /kaggle/working alone is not durable.',
        'Run setup cells in order, choose the stage, then run the dispatch cell. The final export cell can rebuild CSV/JSON reports from committed records after an interruption.',
        'Clean TEST metrics and validation selection are distinct; never use TEST to adjust hyperparameters after viewing these reports.',
        '','## Validation performed','',
        'Local authoring: syntax/AST, nbformat structure and standard-library helper tests only; no local resource downloads, model inference or GPU training.',
        'Kaggle runtime assertions: exact sizes/labels/leakage audit, complete epochs, backbone equality, saved artifact hashes, checkpoint reload, wrapper/recipe preflight, initial-score consistency, all test IDs, unique keys, schemas, denominator/bound arithmetic and paper-difference arithmetic.']
    atomic_text(ROOT/'reproduction_report.md','\n'.join(lines)+'\n')

REPORT_NAMES=['experiment_config.json','dataset_manifest.json','environment_info.json','training_history.csv',
    'training_summary.json','clean_test_predictions.csv','clean_test_metrics.json',
    *[f'attack_results_{m}_{a}.csv' for m,a in ALL_JOBS],
    'summary_metrics.csv','summary_metrics.json','paper_comparison.csv','successful_examples.csv','reproduction_report.md']

def compact_zip():
    target=ROOT/'pure_reproduction_reports.zip'; temp=target.with_suffix('.zip.tmp')
    # Include readable top-level reports and plots; exclude checkpoint weights, RNG ledgers and source.
    with zipfile.ZipFile(temp,'w',compression=zipfile.ZIP_DEFLATED) as z:
        for p in sorted(ROOT.iterdir()):
            if p.is_file() and p.suffix in {'.csv','.json','.md','.txt','.log'} and p.name!='pip_plan.json':
                z.write(p,p.name)
        for p in sorted((ROOT/'plots').glob('*.png')): z.write(p,str(p.relative_to(ROOT)))
    os.replace(temp,target)
    with zipfile.ZipFile(target) as z:
        assert not any(n.endswith(('.safetensors','.bin','.pt','.pth')) for n in z.namelist())
    return target

def save_attack_outputs(make_plots=False):
    assert RUN_FINGERPRINT is not None
    frames={}; summaries=[]; all_rows=[]
    for model_id,attack_name in ALL_JOBS:
        rows,events=read_job(model_id,attack_name)
        inflight=job_directory(model_id,attack_name)/'inflight.json'
        if inflight.exists():
            active=unseal(read_json(inflight)); assert active['run_fingerprint']==RUN_FINGERPRINT
            if active['sequence']>len(events): rows[active['sample_id']]['interrupted_attempt_cost_unknown']=True
        values=[rows[s['sample_id']] for s in TEST_ROWS]
        df=pd.DataFrame(values)
        assert len(df)==1821 and not df.duplicated(['run_fingerprint','model','attack','sample_id']).any()
        required={'original_text','adversarial_text','label','clean_prediction','adversarial_prediction',
            'attack_status','changed_words','num_queries','sample_id','error','attempts','elapsed_seconds'}
        assert required<=set(df.columns)
        assert df.sample_id.tolist()==[r['sample_id'] for r in TEST_ROWS]
        atomic_csv(ROOT/f'attack_results_{model_id}_{attack_name}.csv',df)
        frames[(model_id,attack_name)]=rows; summaries.append(job_metrics(model_id,attack_name,rows)); all_rows+=values
    paired=paired_metrics(frames); comparison=paper_comparison(summaries)
    atomic_csv(ROOT/'summary_metrics.csv',pd.DataFrame(summaries))
    atomic_json(ROOT/'summary_metrics.json',dict(run_fingerprint=RUN_FINGERPRINT,seed=SEED,split='TEST',
        complete=all(r['complete'] for r in summaries),jobs=summaries,paired_common_correct=paired,
        definitions=dict(clean='initially_correct/1821',ASR='successful/initially_correct only at complete coverage',
            AUA='(initially_correct-successful)/1821 only at complete coverage',
            eligible_coverage='(successful+failed)/initially_correct',AvgQ='all attempted final-attempt logical query counts',
            query_units='logical TextAttack victim queries; physical forwards and retry costs recorded separately')))
    atomic_csv(ROOT/'paired_common_correct.csv',pd.DataFrame(paired))
    atomic_csv(ROOT/'paper_comparison.csv',pd.DataFrame(comparison))
    successes=pd.DataFrame([r for r in all_rows if r['attack_status']=='success'],columns=list(all_rows[0]))
    successes['human_label_preservation_review']='not reviewed; automated constraints only'
    atomic_csv(ROOT/'successful_examples.csv',successes)
    write_reproduction_report(summaries,comparison,paired)
    if make_plots: plot_results(summaries,frames)
    EXPERIMENT_CONFIG['status']='COMPLETE_ELIGIBLE_COVERAGE' if all(r['complete'] for r in summaries) else 'INCOMPLETE'
    EXPERIMENT_CONFIG['run_fingerprint']=RUN_FINGERPRINT
    atomic_json(ROOT/'experiment_config.json',EXPERIMENT_CONFIG)
    compact_zip()
    return summaries


def export_current_reports(make_plots=False):
    """Flush committed state at every scientific stage boundary and caught interrupt."""
    if RUN_FINGERPRINT is None and all((ROOT/'clean_records'/m/'complete.json').exists() for m in MODEL_IDS):
        bind_attack_run()
    if RUN_FINGERPRINT is not None:
        return save_attack_outputs(make_plots=make_plots)
    refresh_training_reports()
    write_reproduction_report()
    compact_zip()


## 14. Execute the selected stage

All expensive work is dispatched here, after dependency/data/source preflight and a lightweight check of pinned API signatures, dataset content hashes, atomic output writes and restored checkpoint paths. No optimizer step runs in the preflight; the earlier resource preflight includes only capped synthetic/API diagnostics and model forward/backward/save-reload probes. `train` runs the full training grid and saves both winners. `evaluate` requires winners and computes complete clean TEST and sensitivity diagnostics. `attack` requires both complete clean ledgers and runs the selected jobs. `all` performs all three in order. For Kaggle, prefer separate persisted `train` → `evaluate` → `attack` stages; 18 full candidates and four full attacks can exceed a session. Keep `HYPERPARAM_MODE='paper_grid'` for the faithful grid. Each normal stage boundary and caught interrupt exports current reports regardless of the 100-example periodic interval.

To resume within an existing kernel, leave the scientific configuration unchanged and rerun this cell. To resume a fresh Kaggle session, attach the saved full artifact tree, set stage/job controls, then run setup and dispatch. Repeated execution skips only hash-verified completed candidates and valid resolved attack records; errors and mismatches remain visible. Training or evaluation interruptions preserve their existing checkpoints/clean prefixes.


In [14]:
try:
    with stage('lightweight Kaggle runtime preflight before stage dispatch'):
        runtime_preflight()
    if RUN_STAGE in {'all','train'}:
        with stage('all four epochs for every training candidate'):
            train_models()
        export_current_reports()
    if RUN_STAGE in {'all','evaluate'}:
        with stage('reload both winners and evaluate all 1821 labeled TEST samples'):
            evaluate_models()
        export_current_reports()
    if RUN_STAGE in {'all','attack'}:
        with stage('bind both clean TEST runs before independent attacks'):
            bind_attack_run()
            save_attack_outputs(make_plots=False)  # export pending status before expensive jobs
        for model_id,attack_name in SELECTED_JOBS:
            with stage(f'full TEST attack: {model_id}/{attack_name}'):
                run_attack_job(model_id,attack_name) # its finally always exports, even below 100 commits
except (KeyboardInterrupt,SystemExit):
    LOG.warning('Stage interrupted: exporting every committed result; an unfinished in-flight attempt stays unresolved')
    raise
finally:
    export_current_reports(make_plots=True)


2026-10-31 18:01:47,982 INFO Starting lightweight Kaggle runtime preflight before stage dispatch
2026-10-31 18:01:48,053 INFO Kaggle runtime preflight passed: dependencies, dataset integrity, model/recipe probes, APIs and 0 restored checkpoint paths
2026-10-31 18:01:48,054 WARNING RUN_STAGE=all schedules 18 candidates and four full attacks: high single-session risk. Prefer saved train/evaluate/attack stages; paper_grid remains unchanged.
2026-10-31 18:01:48,057 INFO Starting all four epochs for every training candidate
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at /kaggle/working/.pure_cache/huggingface/hub/models--bert-base-uncased/snapshots/86b5e0934494bd15c9632b12f734a8a67f723594 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
2026-10-31 18:01:49,432 INFO bert_baseline: verified 197 source-learned tensors

Epoch,Training Loss,Validation Loss,Accuracy
1,0.306800,0.317559,0.910550
2,0.312300,0.373144,0.907110
3,0.149200,0.446606,0.908257
4,0.078200,0.503730,0.905963


2026-10-31 18:04:13,214 INFO Committed Trainer checkpoint /kaggle/working/pure_reproduction/candidates/bert_baseline/lr_1e-05_bs_8/checkpoints/checkpoint-865 (epoch=1.0, optimizer step=865)
2026-10-31 18:06:39,488 INFO Committed Trainer checkpoint /kaggle/working/pure_reproduction/candidates/bert_baseline/lr_1e-05_bs_8/checkpoints/checkpoint-1730 (epoch=2.0, optimizer step=1730)
2026-10-31 18:09:05,991 INFO Committed Trainer checkpoint /kaggle/working/pure_reproduction/candidates/bert_baseline/lr_1e-05_bs_8/checkpoints/checkpoint-2595 (epoch=3.0, optimizer step=2595)
2026-10-31 18:11:31,968 INFO Committed Trainer checkpoint /kaggle/working/pure_reproduction/candidates/bert_baseline/lr_1e-05_bs_8/checkpoints/checkpoint-3460 (epoch=4.0, optimizer step=3460)
2026-10-31 18:11:32,097 INFO Trainer completed: reported epoch=4.0 optimizer steps=3460 scheduled=3460 observed sample windows=[6920, 6920, 6920, 6920]
Some weights of BertForSequenceClassification were not initialized from the model 

Epoch,Training Loss,Validation Loss,Accuracy
1,0.297600,0.235156,0.905963
2,0.154100,0.292368,0.902523
3,0.131500,0.359731,0.905963
4,0.151900,0.386964,0.911697


2026-10-31 18:13:52,755 INFO Committed Trainer checkpoint /kaggle/working/pure_reproduction/candidates/bert_baseline/lr_1e-05_bs_16/checkpoints/checkpoint-433 (epoch=1.0, optimizer step=433)
2026-10-31 18:16:03,017 INFO Committed Trainer checkpoint /kaggle/working/pure_reproduction/candidates/bert_baseline/lr_1e-05_bs_16/checkpoints/checkpoint-866 (epoch=2.0, optimizer step=866)
2026-10-31 18:18:13,911 INFO Committed Trainer checkpoint /kaggle/working/pure_reproduction/candidates/bert_baseline/lr_1e-05_bs_16/checkpoints/checkpoint-1299 (epoch=3.0, optimizer step=1299)
2026-10-31 18:20:22,987 INFO Committed Trainer checkpoint /kaggle/working/pure_reproduction/candidates/bert_baseline/lr_1e-05_bs_16/checkpoints/checkpoint-1732 (epoch=4.0, optimizer step=1732)
2026-10-31 18:20:23,118 INFO Trainer completed: reported epoch=4.0 optimizer steps=1732 scheduled=1732 observed sample windows=[6920, 6920, 6920, 6920]
Some weights of BertForSequenceClassification were not initialized from the mode

Epoch,Training Loss,Validation Loss,Accuracy
1,0.297400,0.249928,0.902523
2,0.175700,0.258491,0.904817
3,0.118400,0.299352,0.899083
4,0.092200,0.312199,0.903670


... (39 outputs left out)

bert_pure/textfooler pass 1:   0%|          | 0/1821 [00:00<?, ?it/s]

2026-10-31 22:27:09,697 INFO Fingerprint not found. Saved model loading will continue.
2026-10-31 22:27:09,698 INFO path_and_singleprint metric could not be logged. Saved model loading will continue.
2026-10-31 23:58:24,953 INFO Starting full TEST attack: bert_pure/textbugger
textattack: Unknown if model of class <class 'modeling_bert.PureBertForSequenceClassification'> compatible with goal function <class 'textattack.goal_functions.classification.untargeted_classification.UntargetedClassification'>.


bert_pure/textbugger pass 1:   0%|          | 0/1821 [00:00<?, ?it/s]

2026-10-31 23:58:28,963 INFO Fingerprint not found. Saved model loading will continue.
2026-10-31 23:58:28,964 INFO path_and_singleprint metric could not be logged. Saved model loading will continue.


## 15. Rebuild and validate exports; Kaggle downloads

Run this cell independently after setup to rebuild reports from existing clean and attack ledgers, including interrupted jobs. A stage intentionally stopping after training has no measured clean/attack results. All required final artifacts are asserted once clean results exist. Each of the four per-example CSVs has 1,821 rows, including pending/ineligible/error rows, even when only one job was selected. The compact ZIP contains reports, plots and CSV/JSON files only; model checkpoints remain individually available in Kaggle Outputs.


In [15]:
if RUN_FINGERPRINT is None and all((ROOT/'clean_records'/m/'complete.json').exists() for m in MODEL_IDS):
    bind_attack_run()
if RUN_FINGERPRINT is not None:
    SUMMARY=save_attack_outputs(make_plots=True)
    for name in REPORT_NAMES:
        assert (ROOT/name).is_file(), f'Missing requested final output: {name}'
    for model_id,attack_name in ALL_JOBS:
        frame=pd.read_csv(ROOT/f'attack_results_{model_id}_{attack_name}.csv',keep_default_na=False)
        assert len(frame)==1821 and frame.sample_id.is_unique
        assert set(frame.attack_status)<={'success','failed','ineligible','error','initial_mismatch','pending'}
    assert len(pd.read_csv(ROOT/'clean_test_predictions.csv'))==3642
    assert read_json(ROOT/'clean_test_metrics.json')['split']=='TEST'
    display(pd.read_csv(ROOT/'paper_comparison.csv'))
    display(pd.DataFrame(SUMMARY)[['model','attack','status','N_total','N_initially_correct',
        'N_successful','N_failed','N_error','N_pending','eligible_coverage_pct','asr_pct','aua_pct']])
    if not all(s['complete'] for s in SUMMARY):
        print('INCOMPLETE: definitive full-test ASR/AUA are null until every eligible result is resolved. See bounds and errors.')
    else:
        print('Complete eligible coverage measured. Inspect actual-minus-paper differences before claiming reproduction.')
else:
    refresh_training_reports(); write_reproduction_report(); compact_zip()
    print('Training/setup artifacts only: no completed clean TEST or attack measurements are claimed.')
for name in REPORT_NAMES+['paired_common_correct.csv','inference_diagnostics.json','dataset_overlap_report.json','pure_reproduction_reports.zip']:
    path=ROOT/name
    if path.exists():
        print(f'{name}: {path.stat().st_size:,} bytes')
        display(FileLink(str(path)))
print('Full resumable artifact tree:',ROOT)
print('Save this Kaggle notebook version/output and reattach the full tree as an input dataset for cross-session resume.')


,model,attack,metric,actual,paper_reference,actual_minus_paper,difference_unit,eligible_coverage_pct,complete,note
0,bert_baseline,textfooler,clean_accuracy_pct,91.982427,92.09,-0.107573,percentage points,100.0,True,reference only; never an experimental output s...
1,bert_baseline,textfooler,aua_pct,5.546403,6.32,-0.773597,percentage points,100.0,True,reference only; never an experimental output s...
2,bert_baseline,textfooler,asr_pct,93.970149,93.14,0.830149,percentage points,100.0,True,reference only; never an experimental output s...
3,bert_baseline,textfooler,avg_queries_all_attacked,86.490149,87.07,-0.579851,queries,100.0,True,reference only; never an experimental output s...
4,bert_baseline,textbugger,clean_accuracy_pct,91.982427,92.09,-0.107573,percentage points,100.0,True,reference only; never an experimental output s...
5,bert_baseline,textbugger,aua_pct,31.466227,30.37,1.096227,percentage points,100.0,True,reference only; never an experimental output s...
6,bert_baseline,textbugger,asr_pct,65.791045,67.02,-1.228955,percentage points,100.0,True,reference only; never an experimental output s...
7,bert_baseline,textbugger,avg_queries_all_attacked,40.101493,41.19,-1.088507,queries,100.0,True,reference only; never an experimental output s...
8,bert_pure,textfooler,clean_accuracy_pct,90.939044,90.88,0.059044,percentage points,100.0,True,reference only; never an experimental output s...
9,bert_pure,textfooler,aua_pct,30.642504,30.37,0.272504,percentage points,100.0,True,reference only; never an experimental output s...


,model,attack,status,N_total,N_initially_correct,N_successful,N_failed,N_error,N_pending,eligible_coverage_pct,asr_pct,aua_pct
0,bert_baseline,textfooler,COMPLETE,1821,1675,1574,101,0,0,100.0,93.970149,5.546403
1,bert_baseline,textbugger,COMPLETE,1821,1675,1102,573,0,0,100.0,65.791045,31.466227
2,bert_pure,textfooler,COMPLETE,1821,1656,1098,558,0,0,100.0,66.304348,30.642504
3,bert_pure,textbugger,COMPLETE,1821,1656,753,903,0,0,100.0,45.471014,49.588138


Complete eligible coverage measured. Inspect actual-minus-paper differences before claiming reproduction.
experiment_config.json: 41,653 bytes


/kaggle/working/pure_reproduction/experiment_config.json

dataset_manifest.json: 3,182,639 bytes


/kaggle/working/pure_reproduction/dataset_manifest.json

... (33 outputs left out)

/kaggle/working/pure_reproduction/dataset_overlap_report.json

pure_reproduction_reports.zip: 4,220,917 bytes


/kaggle/working/pure_reproduction/pure_reproduction_reports.zip

Full resumable artifact tree: /kaggle/working/pure_reproduction
Save this Kaggle notebook version/output and reattach the full tree as an input dataset for cross-session resume.


## 16. Interpretation and unresolved paper ambiguities

This notebook is an executable reproduction protocol, **not a claim of successful numerical reproduction before Kaggle execution**. Inspect measured clean metrics, complete coverage, Table 1 differences and stochastic/padding diagnostics together. Select no hyperparameter using TEST or attack results.

Remaining ambiguities include the authors' exact raw SST-2 distribution and preprocessing, seed(s) and run aggregation, library revisions/default recipe versions, tokenization/truncation/padding settings, physical versus effective batching, dropout overrides, optimizer decay/schedule/warmup, epoch selection/ties, inference RNG handling and caching, and the precise AvgQ denominator. These are disclosed with chosen defaults in the config and generated report. Repeated seeds must remain separate experiments before any explicitly labeled aggregate analysis. Errors, resource failures and unresolved initial predictions are evidence to inspect, never evidence of attack resistance.
